## V1

#### 学习建议
- 第1-2天: 模块1-2
- 跑通数据加载部分

#### 熟悉 Pandas 基本操作

- 第3-4天: 模块3-4
- 理解特征工程原理

- 手写一个简单的 Target Encoder

#### 第5-6天: 模块5
- 学习 Optuna 基本用法

- 尝试调参一个简单模型

#### 第7-8天: 模块6-7
- 理解交叉验证

- 跑通完整流程

#### 第9-10天: 整合
- 运行完整代码

- 尝试修改参数观察效果

#### 📝 学习检查清单
- 能解释 seed_everything 的作用

- 能用 Pandas 完成数据筛选、分组、聚合

- 能手写 digit 特征提取

- 能解释 Target Encoding 的平滑

In [3]:
# ============================================
# 🔥 XGBoost + Optuna 自动调参 - 完整本地版
# ============================================

# !pip install optuna xgboost -q

import gc
import random
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from pandas.api.types import is_object_dtype, is_string_dtype, is_categorical_dtype
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb
from xgboost import XGBClassifier
import optuna # 这个是 Optuna 的核心库，用于创建和管理优化研究，以及定义目标函数。主要功能包括：
# - 创建研究（Study）：optuna.create_study() 用于创建一个新的优化研究对象，指定优化方向（maximize 或 minimize）和采样器（如 TPESampler）。
# - 定义目标函数：用户需要定义一个目标函数（objective function），该函数接受一个 trial 对象作为参数，并返回一个数值（如模型的性能指标）。Optuna 将在优化过程中调用这个函数来评估不同的参数组合。
from optuna.samplers import TPESampler # 这个是 Optuna 中的一个采样器，基于 Tree-structured Parzen Estimator (TPE) 算法。它用于在参数空间中智能地选择下一个要评估的参数组合。TPESampler 通过构建一个概率模型来估计不同参数组合的性能，并根据这个模型选择最有可能改进目标函数值的参数组合。

warnings.filterwarnings("ignore")

# ============================================
# 全局配置
# ============================================
SEED = 2026
N_FOLDS = 5
TARGET_COL = "Irrigation_Need"
ID_COL = "id"
N_CLASSES = 3
USE_GPU = False
OPTUNA_TRIALS = 20  # 调参试验次数，可调整

def seed_everything(seed: int = 2026) -> None:
    random.seed(seed)
    np.random.seed(seed)

seed_everything(SEED)
print("✅ Config loaded")

# ============================================
# 加载本地数据
# ============================================
print("\n📂 加载数据...")
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sub = pd.read_csv("sample_submission.csv")

# 如果有额外的原始数据集（sample_submission.csv 中的 Irrigation_Requirement 列）
# 注意：根据你的实际情况，可能需要调整
try:
    original = pd.read_csv("sample_submission.csv").rename(columns={"Irrigation_Requirement": TARGET_COL}) # 尝试加载原始数据集，并重命名目标列以匹配训练数据
    if TARGET_COL in original.columns and not original[TARGET_COL].isnull().all(): # 检查目标列是否存在且不全为缺失值
        original["id"] = range(train["id"].max() + 1, train["id"].max() + 1 + len(original)) # 为原始数据集分配新的 id，确保与训练数据集不冲突
        train = pd.concat([train, original], ignore_index=True) # 将原始数据集合并到训练数据集中
        print(f"  ✅ 合并原始数据集: {len(original)} 条")
except:
    print("  ⚠️ 未找到原始数据集或格式不匹配，跳过")

print(f"  train: {train.shape}")
print(f"  test : {test.shape}")

# 目标编码
label2idx = {"Low": 0, "Medium": 1, "High": 2}
idx2label = {v: k for k, v in label2idx.items()}
train[TARGET_COL] = train[TARGET_COL].map(label2idx).astype("int8")

test_ids = test[ID_COL].copy()
y = train[TARGET_COL].copy()

X_train_raw = train.drop(columns=[TARGET_COL]).copy()
X_test_raw = test.copy()

if ID_COL in X_train_raw.columns:
    X_train_raw = X_train_raw.drop(columns=[ID_COL])
if ID_COL in X_test_raw.columns:
    X_test_raw = X_test_raw.drop(columns=[ID_COL])

print(f"\n📊 目标分布:")
print(y.value_counts(normalize=True).sort_index().round(4))

# ============================================
# 数据预处理
# ============================================
print("\n🔧 数据预处理...")

# 处理缺失值和无穷大
def clean_numeric(df, num_cols):
    df = df.copy()
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
            df[c] = df[c].fillna(df[c].median() if df[c].notna().any() else 0)
            df[c] = df[c].replace([np.inf, -np.inf], 0)
    return df

def get_base_cols(df):
    cats = [c for c in df.columns if is_object_dtype(df[c]) or is_string_dtype(df[c]) or is_categorical_dtype(df[c])]
    nums = [c for c in df.columns if c not in cats]
    return cats, nums

CATS_BASE, NUMS_BASE = get_base_cols(X_train_raw)
print(f"  类别特征: {len(CATS_BASE)}, 数值特征: {len(NUMS_BASE)}")

# 清理数值列
X_train_raw = clean_numeric(X_train_raw, NUMS_BASE)
X_test_raw = clean_numeric(X_test_raw, NUMS_BASE)

# ============================================
# 简化特征工程
# ============================================
def FE_simple(df, M, num_cols):
    """简化版特征工程"""
    out = df.copy()
    
    # 添加 digit 特征
    for c in num_cols:
        if c not in out.columns:
            continue
        for k in [0, 1, 2, -1, -2]:
            col_name = f"{c}_digit{k}"
            try:
                if k >= 0:
                    divisor = 10**k
                    values = (out[c] // divisor) % 10
                else:
                    values = (out[c] * (10**(-k))) % 10
                out[col_name] = values.fillna(0).astype('int8')
            except:
                out[col_name] = 0
    
    # 根据最大值调整精度
    for c in num_cols:
        if c in out.columns:
            if M.get(c, 0) < 10:
                out[c] = out[c].round(3)
            elif M.get(c, 0) < 100:
                out[c] = out[c].round(2)
            else:
                out[c] = out[c].round(1)
    
    return out

M_vals = X_train_raw[NUMS_BASE].max().to_dict()
X_train_fe = FE_simple(X_train_raw, M_vals, NUMS_BASE[:8])  # 只对前8个数值列做，加速
X_test_fe = FE_simple(X_test_raw, M_vals, NUMS_BASE[:8])

# 删除常量列
DROP = [c for c in X_test_fe.columns if X_test_fe[c].nunique() <= 1]
if DROP:
    print(f"  删除常量列: {len(DROP)}")
    X_train_fe.drop(columns=DROP, inplace=True, errors='ignore')
    X_test_fe.drop(columns=DROP, inplace=True, errors='ignore')

# ============================================
# 类别特征编码
# ============================================
print("\n🏷️ 类别特征编码...")

CATEGORY = [c for c in CATS_BASE if c in X_train_fe.columns]
CATEGORY += [c for c in X_test_fe.columns if 'digit' in c]

for c in CATEGORY:
    if c not in X_train_fe.columns:
        continue
    # 频率编码
    freq = X_train_fe[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)
    X_train_fe[c] = X_train_fe[c].map(lambda x: mapping.get(x, mapping_default)).fillna(mapping_default)
    X_test_fe[c] = X_test_fe[c].map(lambda x: mapping.get(x, mapping_default)).fillna(mapping_default)

FEATURES = CATEGORY + [c for c in NUMS_BASE if c in X_test_fe.columns]

print(f"  处理后: train {X_train_fe.shape}, test {X_test_fe.shape}")

# ============================================
# 简化的目标编码
# ============================================
class SimpleTE:
    """简化的目标编码"""
    def __init__(self, smoothing=10):
        self.smoothing = smoothing
        self.maps = {}
        
    def fit(self, X, y, cols):
        self.global_mean = y.mean()
        for c in cols:
            if c not in X.columns:
                continue
            stats = y.groupby(X[c]).agg(['mean', 'count'])
            stats['smooth'] = (stats['count'] * stats['mean'] + self.smoothing * self.global_mean) / (stats['count'] + self.smoothing)
            self.maps[c] = stats['smooth'].to_dict()
        return self
    
    def transform(self, X, cols):
        X = X.copy()
        for c in cols:
            if c in self.maps:
                X[f'{c}_TE'] = X[c].map(self.maps[c]).fillna(self.global_mean).astype(np.float32)
        return X

# ============================================
# 样本权重
# ============================================
unique, counts = np.unique(y, return_counts=True)
weights_dict = {cls: (len(y) / len(unique)) / cnt for cls, cnt in zip(unique, counts)}
sample_weights = np.array([weights_dict[lbl] for lbl in y])

print(f"\n⚖️ 样本权重: Low={weights_dict[0]:.2f}, Medium={weights_dict[1]:.2f}, High={weights_dict[2]:.2f}")

# ============================================
# Optuna 调参
# ============================================
print(f"\n🔍 开始 Optuna 调参 (共 {OPTUNA_TRIALS} 次试验)...")

def objective_xgb(trial):# 定义 xgboost 的目标函数，接受一个 Optuna trial 对象作为参数，并返回一个数值（如模型的性能指标）。Optuna 将在优化过程中调用这个函数来评估不同的参数组合。
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 6),# 这个参数控制树的最大深度，较大的值可以捕捉更复杂的关系，但也可能增加过拟合风险。建议在 3 到 6 之间进行搜索。
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.05),# 这句的意思是不用主动设置精确的学习率，而是让 Optuna 在 0.01 到 0.05 的范围内自动选择一个最佳的学习率。较小的学习率可以提高模型的性能，但也可能需要更多的树（n_estimators）来达到同样的效果。建议在 0.01 到 0.05 之间进行搜索，可以使用对数尺度（log=True）来更有效地探索较大的范围。
        'n_estimators': trial.suggest_int('n_estimators', 1500, 3000), # 不用主动设置精确的树的数量，而是让 Optuna 在 1500 到 3000 的范围内自动选择一个最佳的树的数量。较大的树的数量可以提高模型的性能，但也可能增加训练时间和过拟合风险。建议在 1500 到 3000 之间进行搜索。
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 5), # 这个参数控制每个叶子节点的最小样本权重和，较大的值可以防止过拟合。建议在 1 到 5 之间进行搜索。
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),# 这个参数控制每棵树随机采样的特征比例，较小的值可以增加模型的多样性，帮助防止过拟合。建议在 0.6 到 0.9 之间进行搜索，可以使用对数尺度（log=True）来更有效地探索较大的范围。
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 0.8),# 这个参数控制每棵树随机采样的特征比例，较小的值可以增加模型的多样性，帮助防止过拟合。建议在 0.4 到 0.8 之间进行搜索，可以使用对数尺度（log=True）来更有效地探索较大的范围。
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 1.0, log=True),# 这个参数控制 L1 正则化的强度，较大的值会增加正则化效果，帮助防止过拟合。建议在 1e-5 到 1.0 之间进行搜索，可以使用对数尺度（log=True）来更有效地探索较大的范围。
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0),# 这个参数控制 L2 正则化的强度，较大的值会增加正则化效果，帮助防止过拟合。建议在 1.0 到 10.0 之间进行搜索，可以使用对数尺度（log=True）来更有效地探索较大的范围。
        'gamma': trial.suggest_float('gamma', 0.0, 0.2), # 这个参数控制树的分裂所需的最小损失减少，较大的值可以防止过拟合。建议在 0.0 到 0.2 之间进行搜索。
        'tree_method': 'hist', # 这个参数控制树的构建方法，'hist' 是一种基于直方图的高效算法，适用于大规模数据集。使用 'hist' 可以显著加快训练速度，尤其是在 CPU 上。
        'device': 'cuda' if USE_GPU else 'cpu', # 这个参数控制训练设备，如果你有 NVIDIA GPU 并且安装了支持 GPU 的 XGBoost 版本，设置为 'cuda' 可以显著加快训练速度。否则，设置为 'cpu'。
        'random_state': SEED,
        'n_jobs': -1, # 使用所有 CPU 核心加速训练
        'enable_categorical': True,
        'objective': 'multi:softprob',
        'num_class': N_CLASSES,
        'eval_metric': 'mlogloss',
    }
    
    kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED) # 这个参数控制交叉验证的折数，较小的值可以加快调参速度，但可能导致评估结果不稳定。建议在 3 到 5 之间进行搜索。
    scores = [] # 这个列表用于存储每次交叉验证的性能指标（如平衡准确率）。在每次折叠中，模型会在训练集上训练，并在验证集上评估性能，计算得到的指标会被添加到这个列表中。最后，函数会返回这些指标的平均值，作为当前参数组合的性能评估结果。
    
    for tr_idx, va_idx in kf.split(X_train_fe, y): # 这个循环遍历交叉验证的每个折叠，tr_idx 和 va_idx 分别是训练集和验证集的索引。对于每个折叠，模型会在训练集上训练，并在验证集上评估性能。
        X_tr, X_va = X_train_fe.iloc[tr_idx].copy(), X_train_fe.iloc[va_idx].copy() # 这个代码行从训练数据集中根据 tr_idx 和 va_idx 索引分别创建训练集 X_tr 和验证集 X_va 的副本。使用 .copy() 是为了确保在后续的处理过程中对 X_tr 和 X_va 的修改不会影响原始的 X_train_fe 数据。
        y_tr, y_va = y.iloc[tr_idx].copy(), y.iloc[va_idx].copy() # 这个代码行从目标变量 y 中根据 tr_idx 和 va_idx 索引分别创建训练集 y_tr 和验证集 y_va 的副本。使用 .copy() 是为了确保在后续的处理过程中对 y_tr 和 y_va 的修改不会影响原始的 y 数据。
        w_tr = sample_weights[tr_idx] # 这个代码行从样本权重数组 sample_weights 中根据 tr_idx 索引创建训练集的样本权重 w_tr 的副本。这样，在训练模型时，可以使用 w_tr 来为训练集中的每个样本分配不同的权重，以处理类别不平衡问题。

        # 目标编码
        te = SimpleTE(smoothing=10) # 这个代码行创建了一个 SimpleTE 类的实例 te，并设置平滑参数 smoothing 为 10。SimpleTE 是一个简化的目标编码类，用于将类别特征转换为数值特征，以便模型能够更好地处理这些特征。在后续的代码中，te 将被用于对训练集和验证集中的类别特征进行目标编码。
        te.fit(X_tr, y_tr, FEATURES) # 这个代码行调用 SimpleTE 实例 te 的 fit 方法，对训练集 X_tr 和目标变量 y_tr 进行拟合，并指定要编码的特征列表 FEATURES。fit 方法会计算每个类别特征在训练集中的统计信息（如均值和计数），并根据这些统计信息创建一个映射，用于将类别特征转换为数值特征。这个过程是目标编码的一部分，可以帮助模型更好地理解类别特征与目标变量之间的关系。
        X_tr_enc = te.transform(X_tr, FEATURES) # 这个代码行调用 SimpleTE 实例 te 的 transform 方法，对训练集 X_tr 中的类别特征进行目标编码，并返回编码后的训练集 X_tr_enc。transform 方法会使用在 fit 阶段计算的映射，将类别特征转换为数值特征。这个过程是目标编码的一部分，可以帮助模型更好地理解类别特征与目标变量之间的关系。
        X_va_enc = te.transform(X_va, FEATURES) # 这个代码行调用 SimpleTE 实例 te 的 transform 方法，对验证集 X_va 中的类别特征进行目标编码，并返回编码后的验证集 X_va_enc。transform 方法会使用在 fit 阶段计算的映射，将类别特征转换为数值特征。这个过程是目标编码的一部分，可以帮助模型更好地理解类别特征与目标变量之间的关系。
        
        # 确保列一致
        common_cols = [c for c in X_tr_enc.columns if c in X_va_enc.columns] # 这个代码行创建了一个列表 common_cols，其中包含了 X_tr_enc 和 X_va_enc 两个数据集中都存在的列名。这个步骤是为了确保在后续的模型训练和评估过程中，训练集和验证集使用的特征列是一致的，以避免因列不匹配而导致的错误。,这句代码的意思是：在训练集编码后的数据 X_tr_enc 和验证集编码后的数据 X_va_enc 中，找出两者都包含的列名，并将这些列名存储在 common_cols 列表中。这样，在后续的模型训练和评估过程中，可以确保使用的特征列是一致的。
        X_tr_enc = X_tr_enc[common_cols]
        X_va_enc = X_va_enc[common_cols]
        
        model = XGBClassifier(**params)
        model.fit(X_tr_enc, y_tr, sample_weight=w_tr, verbose=False)
        va_p = model.predict_proba(X_va_enc)
        scores.append(balanced_accuracy_score(y_va, va_p.argmax(axis=1)))
        
        del model
        gc.collect()
        
    return np.mean(scores)

study_xgb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED))
optuna.logging.set_verbosity(optuna.logging.WARNING)
study_xgb.optimize(objective_xgb, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

best_xgb_params = study_xgb.best_params
best_xgb_params['tree_method'] = 'hist'
best_xgb_params['device'] = 'cuda' if USE_GPU else 'cpu'
best_xgb_params['random_state'] = SEED
best_xgb_params['n_jobs'] = -1
best_xgb_params['enable_categorical'] = True
best_xgb_params['objective'] = 'multi:softprob'
best_xgb_params['num_class'] = N_CLASSES
best_xgb_params['eval_metric'] = 'mlogloss'

print(f"\n✅ 最佳参数: {best_xgb_params}")
print(f"✅ 最佳 CV 分数: {study_xgb.best_value:.5f}")

# ============================================
# 完整 CV 训练
# ============================================
print(f"\n🚀 开始 {N_FOLDS} 折 CV 训练...")

kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_probs = np.zeros((len(y), N_CLASSES), dtype=np.float32)
test_probs = np.zeros((len(X_test_fe), N_CLASSES), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train_fe, y), start=1):
    print(f"\n📁 Fold {fold}/{N_FOLDS}")
    
    X_tr, X_va = X_train_fe.iloc[tr_idx].copy(), X_train_fe.iloc[va_idx].copy()
    y_tr, y_va = y.iloc[tr_idx].copy(), y.iloc[va_idx].copy()
    w_tr = sample_weights[tr_idx]

    te = SimpleTE(smoothing=10)
    te.fit(X_tr, y_tr, FEATURES)
    X_tr_enc = te.transform(X_tr, FEATURES)
    X_va_enc = te.transform(X_va, FEATURES)
    X_te_enc = te.transform(X_test_fe, FEATURES)
    
    # 确保列一致
    common_cols = [c for c in X_tr_enc.columns if c in X_va_enc.columns and c in X_te_enc.columns]
    X_tr_enc = X_tr_enc[common_cols]
    X_va_enc = X_va_enc[common_cols]
    X_te_enc = X_te_enc[common_cols]
    
    model = XGBClassifier(**best_xgb_params)
    model.fit(X_tr_enc, y_tr, sample_weight=w_tr, verbose=False)
    
    oof_probs[va_idx] = model.predict_proba(X_va_enc)
    test_probs += model.predict_proba(X_te_enc) / N_FOLDS
    
    fold_ba = balanced_accuracy_score(y_va, oof_probs[va_idx].argmax(axis=1))
    print(f"  Fold {fold} BA: {fold_ba:.5f}")
    
    del model
    gc.collect()

# ============================================
# 结果与提交
# ============================================
oof_ba = balanced_accuracy_score(y, oof_probs.argmax(axis=1))
print(f"\n📊 OOF Balanced Accuracy: {oof_ba:.5f}")

# 类别权重微调（可选）
print("\n⚖️ 类别权重微调...")
best_ba = oof_ba
best_w = np.array([1.0, 1.0, 1.0])
for w0 in [0.9, 1.0, 1.1]:
    for w1 in [0.9, 1.0, 1.1]:
        for w2 in [0.9, 1.0, 1.1, 1.2, 1.5]:
            w = np.array([w0, w1, w2])
            adj_probs = oof_probs * w
            adj_probs = adj_probs / adj_probs.sum(axis=1, keepdims=True)
            ba = balanced_accuracy_score(y, adj_probs.argmax(axis=1))
            if ba > best_ba:
                best_ba = ba
                best_w = w

print(f"  最优权重: {best_w.round(4)}")
print(f"  优化后 OOF BA: {best_ba:.5f}")

# 应用权重
test_probs_adj = test_probs * best_w
test_probs_adj = test_probs_adj / test_probs_adj.sum(axis=1, keepdims=True)
test_preds = test_probs_adj.argmax(axis=1)

# 生成提交文件
submission = pd.DataFrame({
    'id': test_ids.values,
    TARGET_COL: [idx2label[p] for p in test_preds]
})

submission.to_csv('submission_optuna_20260416.csv', index=False)
print("\n✅ 提交文件已生成: submission_optuna.csv")
print("\n📈 预测分布:")
print(submission[TARGET_COL].value_counts())

# ============================================
# 特征重要性
# ============================================
print("\n📊 特征重要性 (Top 15):")
te_full = SimpleTE(smoothing=10)
te_full.fit(X_train_fe, y, FEATURES)
X_full = te_full.transform(X_train_fe, FEATURES)
common_cols = [c for c in X_full.columns if c in te_full.transform(X_test_fe, FEATURES).columns]
X_full = X_full[common_cols]

fi_model = XGBClassifier(**best_xgb_params)
fi_model.fit(X_full, y, verbose=False)

importance_df = pd.DataFrame({
    'feature': X_full.columns,
    'importance': fi_model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

print(importance_df.to_string(index=False))

print("\n🎉 全部完成！")

✅ Config loaded

📂 加载数据...
  ✅ 合并原始数据集: 270000 条
  train: (900000, 21)
  test : (270000, 20)

📊 目标分布:
Irrigation_Need
0    0.7110
1    0.2656
2    0.0233
Name: proportion, dtype: float64

🔧 数据预处理...
  类别特征: 8, 数值特征: 11
  删除常量列: 10

🏷️ 类别特征编码...
  处理后: train (900000, 49), test (270000, 49)

⚖️ 样本权重: Low=0.47, Medium=1.25, High=14.28

🔍 开始 Optuna 调参 (共 20 次试验)...


  0%|          | 0/20 [00:00<?, ?it/s]


✅ 最佳参数: {'max_depth': 3, 'learning_rate': 0.028818045864514162, 'n_estimators': 2524, 'min_child_weight': 5, 'subsample': 0.8048761116548554, 'colsample_bytree': 0.6081146493734025, 'reg_alpha': 0.0021628569308857833, 'reg_lambda': 3.316605400985784, 'gamma': 0.10160565613996671, 'tree_method': 'hist', 'device': 'cpu', 'random_state': 2026, 'n_jobs': -1, 'enable_categorical': True, 'objective': 'multi:softprob', 'num_class': 3, 'eval_metric': 'mlogloss'}
✅ 最佳 CV 分数: 0.97557

🚀 开始 5 折 CV 训练...

📁 Fold 1/5
  Fold 1 BA: 0.97700

📁 Fold 2/5
  Fold 2 BA: 0.97419

📁 Fold 3/5
  Fold 3 BA: 0.97723

📁 Fold 4/5
  Fold 4 BA: 0.97617

📁 Fold 5/5
  Fold 5 BA: 0.97542

📊 OOF Balanced Accuracy: 0.97600

⚖️ 类别权重微调...
  最优权重: [1.1 1.  1.5]
  优化后 OOF BA: 0.97692

✅ 提交文件已生成: submission_optuna.csv

📈 预测分布:
Irrigation_Need
Low       158930
Medium    100732
High       10338
Name: count, dtype: int64

📊 特征重要性 (Top 15):
                feature  importance
   Crop_Growth_Stage_TE    0.171827
      Crop_Growth

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
SEED = 42
np.random.seed(SEED)

# 加载数据
print("加载数据...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f"训练集: {train.shape}")
print(f"测试集: {test.shape}")

# 简单的特征工程
def simple_feature_engineering(df):
    df_new = df.copy()
    
    # 分离数值和类别列
    numeric_cols = df_new.select_dtypes(include=[np.number]).columns.tolist()
    if 'id' in numeric_cols:
        numeric_cols.remove('id')
    
    categorical_cols = df_new.select_dtypes(include=['object']).columns.tolist()
    if 'Irrigation_Need' in categorical_cols:
        categorical_cols.remove('Irrigation_Need')
    
    # 添加交叉统计特征（简单有效）
    if len(numeric_cols) > 0:
        df_new['num_sum'] = df_new[numeric_cols].sum(axis=1)
        df_new['num_mean'] = df_new[numeric_cols].mean(axis=1)
        df_new['num_std'] = df_new[numeric_cols].std(axis=1)
    
    # 编码类别特征
    for col in categorical_cols:
        df_new[col] = df_new[col].astype('category').cat.codes
    
    return df_new, numeric_cols, categorical_cols

# 应用特征工程
print("特征工程...")
train_fe, num_cols, cat_cols = simple_feature_engineering(train)
test_fe, _, _ = simple_feature_engineering(test)

# 准备数据
X = train_fe.drop(['id', 'Irrigation_Need'], axis=1)
y = train_fe['Irrigation_Need']
X_test = test_fe.drop(['id'], axis=1)

# 确保列一致
for col in X.columns:
    if col not in X_test.columns:
        X_test[col] = 0
X_test = X_test[X.columns]

# 编码目标变量
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print(f"特征数量: {X.shape[1]}")
print(f"训练样本: {X.shape[0]}")
print(f"测试样本: {X_test.shape[0]}")

# 5折交叉验证
N_FOLDS = 5
kfold = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# XGBoost参数（优化版）
xgb_params = {
    'n_estimators': 1000,
    'max_depth': 6,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1,
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'random_state': SEED,
    'n_jobs': -1,
    'verbosity': 0
}

# LightGBM参数（优化版）
lgb_params = {
    'n_estimators': 1000,
    'num_leaves': 31,
    'max_depth': 6,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1,
    'min_child_samples': 20,
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    'random_state': SEED,
    'n_jobs': -1,
    'verbose': -1
}

# 存储预测
xgb_oof = np.zeros((len(X), 3))
xgb_test = np.zeros((len(X_test), 3))
lgb_oof = np.zeros((len(X), 3))
lgb_test = np.zeros((len(X_test), 3))

# 训练XGBoost
print("\n训练 XGBoost...")
for fold, (train_idx, val_idx) in enumerate(kfold.split(X, y_encoded)):
    print(f"Fold {fold+1}/{N_FOLDS}")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_train, y_train)
    
    xgb_oof[val_idx] = model.predict_proba(X_val)
    xgb_test += model.predict_proba(X_test) / N_FOLDS
    
    score = balanced_accuracy_score(y_val, np.argmax(xgb_oof[val_idx], axis=1))
    print(f"  Score: {score:.4f}")

# 训练LightGBM
print("\n训练 LightGBM...")
for fold, (train_idx, val_idx) in enumerate(kfold.split(X, y_encoded)):
    print(f"Fold {fold+1}/{N_FOLDS}")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_train, y_train)
    
    lgb_oof[val_idx] = model.predict_proba(X_val)
    lgb_test += model.predict_proba(X_test) / N_FOLDS
    
    score = balanced_accuracy_score(y_val, np.argmax(lgb_oof[val_idx], axis=1))
    print(f"  Score: {score:.4f}")

# 计算分数
xgb_score = balanced_accuracy_score(y_encoded, np.argmax(xgb_oof, axis=1))
lgb_score = balanced_accuracy_score(y_encoded, np.argmax(lgb_oof, axis=1))

print(f"\n{'='*40}")
print(f"XGBoost 分数: {xgb_score:.4f}")
print(f"LightGBM 分数: {lgb_score:.4f}")

# 简单平均集成
ensemble_test = (xgb_test + lgb_test) / 2

# 生成提交
final_preds = np.argmax(ensemble_test, axis=1)
final_labels = le.inverse_transform(final_preds)

submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': final_labels
})

submission.to_csv('submission_final.csv', index=False)

print(f"\n提交文件已保存: submission_final.csv")
print(f"预测分布:\n{submission['Irrigation_Need'].value_counts()}")
print(f"\n{'='*40}")
print("完成！可以提交 submission_final.csv 到 Kaggle")

加载数据...
训练集: (630000, 21)
测试集: (270000, 20)
特征工程...
特征数量: 22
训练样本: 630000
测试样本: 270000

训练 XGBoost...
Fold 1/5
  Score: 0.9610
Fold 2/5
  Score: 0.9630
Fold 3/5
  Score: 0.9620
Fold 4/5
  Score: 0.9623
Fold 5/5
  Score: 0.9611

训练 LightGBM...
Fold 1/5
  Score: 0.9613
Fold 2/5
  Score: 0.9632
Fold 3/5
  Score: 0.9624
Fold 4/5
  Score: 0.9616
Fold 5/5
  Score: 0.9623

XGBoost 分数: 0.9619
LightGBM 分数: 0.9622

提交文件已保存: submission_final.csv
预测分布:
Irrigation_Need
Low       159864
Medium    101509
High        8627
Name: count, dtype: int64

完成！可以提交 submission_final.csv 到 Kaggle


## v3 = 

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

# 设置随机种子
SEED = 42
np.random.seed(SEED)

# ========================
# 1. 加载数据
# ========================
print("加载数据...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"训练集: {train.shape}")
print(f"测试集: {test.shape}")

# ========================
# 2. 特征工程（修复版）
# ========================
def feature_engineering(df, fit_encoders=False, encoders=None):
    """增强版特征工程 - 修复类别特征问题"""
    df_new = df.copy()
    
    # 分离数值和类别列
    numeric_cols = df_new.select_dtypes(include=[np.number]).columns.tolist()
    if 'id' in numeric_cols:
        numeric_cols.remove('id')
    
    categorical_cols = df_new.select_dtypes(include=['object']).columns.tolist()
    if 'Irrigation_Need' in categorical_cols:
        categorical_cols.remove('Irrigation_Need')
    
    print(f"数值特征: {len(numeric_cols)}")
    print(f"类别特征: {len(categorical_cols)}")
    
    # 1. 基础统计特征
    if len(numeric_cols) > 0:
        df_new['num_sum'] = df_new[numeric_cols].sum(axis=1)
        df_new['num_mean'] = df_new[numeric_cols].mean(axis=1)
        df_new['num_std'] = df_new[numeric_cols].std(axis=1)
        df_new['num_max'] = df_new[numeric_cols].max(axis=1)
        df_new['num_min'] = df_new[numeric_cols].min(axis=1)
        df_new['num_range'] = df_new['num_max'] - df_new['num_min']
    
    # 2. 交互特征（前5个最重要数值特征的组合）
    if len(numeric_cols) >= 2:
        top_cols = numeric_cols[:min(5, len(numeric_cols))]
        for i in range(len(top_cols)):
            for j in range(i+1, len(top_cols)):
                col1, col2 = top_cols[i], top_cols[j]
                df_new[f'{col1}_x_{col2}'] = df_new[col1] * df_new[col2]
                df_new[f'{col1}_div_{col2}'] = df_new[col1] / (df_new[col2] + 1e-8)
    
    # 3. 多项式特征（平方）
    for col in numeric_cols[:3]:
        df_new[f'{col}_squared'] = df_new[col] ** 2
    
    # 4. 关键修复：编码所有类别特征为数值（供XGB/LGB使用）
    if fit_encoders:
        encoders = {}
    
    for col in categorical_cols:
        df_new[col] = df_new[col].astype(str)
        
        if fit_encoders:
            # 训练时：fit并transform
            le = LabelEncoder()
            df_new[col] = le.fit_transform(df_new[col])
            encoders[col] = le
        else:
            # 测试时：只用transform
            if col in encoders:
                # 处理测试集中未见过的类别
                df_new[col] = df_new[col].map(lambda x: encoders[col].transform([x])[0] 
                                              if x in encoders[col].classes_ else -1)
            else:
                df_new[col] = 0
    
    # 保存类别列名（用于CatBoost）
    cat_cols_for_catboost = categorical_cols.copy()
    
    return df_new, numeric_cols, cat_cols_for_catboost, encoders if fit_encoders else None

# 应用特征工程
print("\n特征工程...")
train_fe, num_cols, cat_cols, encoders = feature_engineering(train, fit_encoders=True)
test_fe, _, _, _ = feature_engineering(test, fit_encoders=False, encoders=encoders)

# 准备数据
X = train_fe.drop(['id', 'Irrigation_Need'], axis=1)
y = train_fe['Irrigation_Need']
X_test = test_fe.drop(['id'], axis=1)

# 确保列一致
for col in X.columns:
    if col not in X_test.columns:
        X_test[col] = 0
X_test = X_test[X.columns]

# 确保所有特征都是数值类型
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(0)

# 编码目标变量
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)

print(f"\n最终特征数量: {X.shape[1]}")
print(f"训练样本: {X.shape[0]}")
print(f"测试样本: {X_test.shape[0]}")
print(f"所有特征类型: {X.dtypes.value_counts()}")

# ========================
# 3. 模型参数
# ========================

# XGBoost参数
xgb_params = {
    'n_estimators': 800,
    'max_depth': 6,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1,
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'random_state': SEED,
    'n_jobs': -1,
    'verbosity': 0
}

# LightGBM参数
lgb_params = {
    'n_estimators': 800,
    'num_leaves': 31,
    'max_depth': 6,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1,
    'min_child_samples': 20,
    'objective': 'multiclass',
    'num_class': 3,
    'metric': 'multi_logloss',
    'random_state': SEED,
    'n_jobs': -1,
    'verbose': -1
}

# CatBoost参数（使用原始字符串类别）
cat_params = {
    'iterations': 800,
    'learning_rate': 0.03,
    'depth': 6,
    'cat_features': cat_cols,  # 直接指定原始类别列名
    'loss_function': 'MultiClass',
    'eval_metric': 'MultiClass',
    'random_seed': SEED,
    'verbose': False,
    'early_stopping_rounds': 50
}

# ========================
# 4. 交叉验证设置
# ========================
N_FOLDS = 5
kfold = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# 存储预测
xgb_oof = np.zeros((len(X), 3))
xgb_test = np.zeros((len(X_test), 3))
lgb_oof = np.zeros((len(X), 3))
lgb_test = np.zeros((len(X_test), 3))
cat_oof = np.zeros((len(X), 3))
cat_test = np.zeros((len(X_test), 3))

# ========================
# 5. 训练 XGBoost
# ========================
print("\n" + "="*50)
print("训练 XGBoost")
print("="*50)

for fold, (train_idx, val_idx) in enumerate(kfold.split(X, y_encoded)):
    print(f"Fold {fold+1}/{N_FOLDS}")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_train, y_train)
    
    xgb_oof[val_idx] = model.predict_proba(X_val)
    xgb_test += model.predict_proba(X_test) / N_FOLDS
    
    score = balanced_accuracy_score(y_val, np.argmax(xgb_oof[val_idx], axis=1))
    print(f"  Score: {score:.4f}")

# ========================
# 6. 训练 LightGBM
# ========================
print("\n" + "="*50)
print("训练 LightGBM")
print("="*50)

for fold, (train_idx, val_idx) in enumerate(kfold.split(X, y_encoded)):
    print(f"Fold {fold+1}/{N_FOLDS}")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_train, y_train)
    
    lgb_oof[val_idx] = model.predict_proba(X_val)
    lgb_test += model.predict_proba(X_test) / N_FOLDS
    
    score = balanced_accuracy_score(y_val, np.argmax(lgb_oof[val_idx], axis=1))
    print(f"  Score: {score:.4f}")

# ========================
# 7. 训练 CatBoost（使用原始数据，保留字符串）
# ========================
print("\n" + "="*50)
print("训练 CatBoost")
print("="*50)

# 为CatBoost准备数据（保留字符串类别）
X_cat = train_fe.copy()
X_cat = X_cat.drop(['id', 'Irrigation_Need'], axis=1)
# 恢复字符串格式（从编码后的数值转回字符串）
for col in cat_cols:
    if col in X_cat.columns:
        # 简单处理：转回字符串
        X_cat[col] = X_cat[col].astype(str)
X_test_cat = X_test.copy()
for col in cat_cols:
    if col in X_test_cat.columns:
        X_test_cat[col] = X_test_cat[col].astype(str)

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_cat, y_encoded)):
    print(f"Fold {fold+1}/{N_FOLDS}")
    X_train, X_val = X_cat.iloc[train_idx], X_cat.iloc[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]
    
    model = CatBoostClassifier(**cat_params)
    model.fit(X_train, y_train, verbose=False)
    
    cat_oof[val_idx] = model.predict_proba(X_val)
    cat_test += model.predict_proba(X_test_cat) / N_FOLDS
    
    score = balanced_accuracy_score(y_val, np.argmax(cat_oof[val_idx], axis=1))
    print(f"  Score: {score:.4f}")

# ========================
# 8. 评估各模型
# ========================
print("\n" + "="*50)
print("模型评估")
print("="*50)

xgb_score = balanced_accuracy_score(y_encoded, np.argmax(xgb_oof, axis=1))
lgb_score = balanced_accuracy_score(y_encoded, np.argmax(lgb_oof, axis=1))
cat_score = balanced_accuracy_score(y_encoded, np.argmax(cat_oof, axis=1))

print(f"XGBoost 分数:  {xgb_score:.4f}")
print(f"LightGBM 分数: {lgb_score:.4f}")
print(f"CatBoost 分数: {cat_score:.4f}")

# ========================
# 9. 简单集成（避免过拟合）
# ========================
print("\n" + "="*50)
print("生成提交文件")
print("="*50)

# 尝试不同的集成方式
# 方式1: 简单平均
ensemble_simple = (xgb_test + lgb_test) / 2
final_preds = np.argmax(ensemble_simple, axis=1)
final_labels = le_target.inverse_transform(final_preds)

submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': final_labels
})

submission.to_csv('submission_final.csv', index=False)
print("提交文件已保存: submission_final.csv")

print(f"\n预测分布:")
print(submission['Irrigation_Need'].value_counts())

print("\n" + "="*50)
print("完成！提交 submission_final.csv 到 Kaggle")
print("="*50)

加载数据...
训练集: (630000, 21)
测试集: (270000, 20)

特征工程...
数值特征: 11
类别特征: 8
数值特征: 11
类别特征: 8

最终特征数量: 48
训练样本: 630000
测试样本: 270000
所有特征类型: float64    40
int64       8
Name: count, dtype: int64

训练 XGBoost
Fold 1/5
  Score: 0.9609
Fold 2/5
  Score: 0.9626
Fold 3/5
  Score: 0.9620
Fold 4/5
  Score: 0.9618
Fold 5/5
  Score: 0.9607

训练 LightGBM
Fold 1/5
  Score: 0.9617
Fold 2/5
  Score: 0.9627
Fold 3/5
  Score: 0.9631
Fold 4/5
  Score: 0.9610
Fold 5/5
  Score: 0.9612

训练 CatBoost
Fold 1/5
  Score: 0.9591
Fold 2/5
  Score: 0.9624
Fold 3/5
  Score: 0.9615
Fold 4/5
  Score: 0.9607
Fold 5/5
  Score: 0.9594

模型评估
XGBoost 分数:  0.9616
LightGBM 分数: 0.9620
CatBoost 分数: 0.9606

生成提交文件
提交文件已保存: submission_final.csv

预测分布:
Irrigation_Need
Low       159865
Medium    101510
High        8625
Name: count, dtype: int64

完成！提交 submission_final.csv 到 Kaggle


## v4=0.97305

In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# ========================
# 1. 设置随机种子
# ========================
SEED = 42
np.random.seed(SEED)

# ========================
# 2. 自定义评估指标
# ========================
def balanced_accuracy_custom(y_true, y_pred):
    """计算平衡准确率"""
    if len(y_pred.shape) == 2:
        y_pred = np.argmax(y_pred, axis=1)
    
    n_classes = 3
    acc_per_class = 0
    for i in range(n_classes):
        mask = y_true == i
        if mask.sum() > 0:
            acc_per_class += np.sum((y_true[mask] == y_pred[mask])) / mask.sum()
    
    return acc_per_class / n_classes

def lgb_eval_metric(y_true, y_pred):
    y_true = y_true.astype(int)
    y_pred = np.argmax(y_pred.reshape(-1, 3), axis=1)
    score = balanced_accuracy_custom(y_true, y_pred)
    return 'balanced_acc', score, True

# ========================
# 3. 加载数据
# ========================
print("加载数据...")
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"训练集: {train.shape}")
print(f"测试集: {test.shape}")

target_col = 'Irrigation_Need'
drop_cols = ['id']

# 编码目标变量
target2idx = {v: i for i, v in enumerate(train[target_col].unique())}
idx2target = {v: i for i, v in target2idx.items()}
train[target_col] = train[target_col].map(target2idx)

# ========================
# 4. 识别特征类型
# ========================
CATS = [c for c in test.columns if train[c].dtype == object and c != target_col]
NUMS = [c for c in test.columns if c not in CATS and c not in drop_cols and c != target_col]

print(f"数值特征: {len(NUMS)}")
print(f"类别特征: {len(CATS)}")

# ========================
# 5. 银牌方案核心：数字根特征
# ========================
def add_digit_features(df, numeric_cols):
    """添加数字根特征 - 这是达到0.979的关键"""
    df_new = df.copy()
    for c in numeric_cols:
        if c not in df_new.columns:
            continue
        # 避免对id列操作
        if c == 'id':
            continue
        for k in range(-4, 5):
            try:
                df_new[f"{c}_digit{k}"] = (df_new[c] // (10**k) % 10).astype('int8')
            except:
                pass
    return df_new

print("\n添加数字根特征...")
train = add_digit_features(train, NUMS)
test = add_digit_features(test, NUMS)

# ========================
# 6. 删除单一值特征
# ========================
DROP = []
for c in test.columns:
    if test[c].nunique() == 1:
        DROP.append(c)

print(f"删除单一值特征: {len(DROP)}")
train = train.drop(DROP, axis=1)
test = test.drop(DROP, axis=1)

# ========================
# 7. 准备特征列表
# ========================
# 更新特征类型
CATS = [c for c in test.columns if train[c].dtype == object and c != target_col]
NUMS = [c for c in test.columns if c not in CATS and c != target_col]

# 类别特征包括：原始类别 + 数字根特征
CATEGORY_FEATURES = CATS + [c for c in test.columns if 'digit' in c and c != target_col]

print(f"\n最终特征数: {len(NUMS) + len(CATEGORY_FEATURES)}")
print(f"其中类别特征: {len(CATEGORY_FEATURES)}")

# ========================
# 8. 准备训练数据
# ========================
X = train.drop([target_col], axis=1)
y = train[target_col].values
X_test = test.copy()

print(f"训练特征形状: {X.shape}")
print(f"测试特征形状: {X_test.shape}")

# ========================
# 9. 计算样本权重（处理类别不平衡）
# ========================
unique, counts = np.unique(y, return_counts=True)
avg_count = len(y) / len(unique)
class_weights = {cls: avg_count / cnt for cls, cnt in zip(unique, counts)}
sample_weights = np.array([class_weights[y_i] for y_i in y])

print(f"类别权重: {class_weights}")

# ========================
# 10. LightGBM参数（银牌方案原版）
# ========================
lgb_params = {
    "n_estimators": 3000,  # 减少到3000加快速度
    'boosting_type': 'gbdt',
    'max_depth': 4,
    'num_leaves': 32,
    'learning_rate': 0.05,
    'feature_fraction': 0.6,
    'bagging_fraction': 0.7,
    'bagging_freq': 1,
    'lambda_l1': 10,
    'lambda_l2': 10,
    'min_child_samples': 12,
    'random_state': SEED,
    'n_jobs': -1,
    'max_bin': 15000,
    'verbosity': -1,
    'subsample': 0.5,
    'subsample_for_bin': 100000,
    'subsample_freq': 1,
}

# ========================
# 11. 训练（使用简单编码）
# ========================
print("\n" + "="*50)
print("训练 LightGBM")
print("="*50)

N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_preds = np.zeros((len(X), 3))
test_preds = np.zeros((len(X_test), 3))

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\nFold {fold+1}/{N_FOLDS}")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    train_weights = sample_weights[train_idx]
    
    # 对类别特征进行简单编码（使用频率编码）
    X_train_encoded = X_train.copy()
    X_val_encoded = X_val.copy()
    X_test_encoded = X_test.copy()
    
    for cat_col in CATEGORY_FEATURES:
        if cat_col not in X_train.columns:
            continue
        
        # 使用频率编码（比Target Encoding简单且稳定）
        freq_map = X_train[cat_col].value_counts(normalize=True).to_dict()
        
        # 应用编码
        X_train_encoded[cat_col + '_freq'] = X_train[cat_col].map(freq_map).fillna(0)
        X_val_encoded[cat_col + '_freq'] = X_val[cat_col].map(freq_map).fillna(0)
        X_test_encoded[cat_col + '_freq'] = X_test[cat_col].map(freq_map).fillna(0)
        
        # 删除原始列
        X_train_encoded = X_train_encoded.drop(cat_col, axis=1)
        X_val_encoded = X_val_encoded.drop(cat_col, axis=1)
        X_test_encoded = X_test_encoded.drop(cat_col, axis=1)
    
    # 确保所有列都是数值类型
    for col in X_train_encoded.columns:
        X_train_encoded[col] = pd.to_numeric(X_train_encoded[col], errors='coerce').fillna(0)
        X_val_encoded[col] = pd.to_numeric(X_val_encoded[col], errors='coerce').fillna(0)
        X_test_encoded[col] = pd.to_numeric(X_test_encoded[col], errors='coerce').fillna(0)
    
    # 训练模型
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_train_encoded, y_train,
        sample_weight=train_weights,
        eval_set=[(X_val_encoded, y_val)],
        eval_metric=lgb_eval_metric,
        callbacks=[lgb.early_stopping(200), lgb.log_evaluation(100)]
    )
    
    # 预测
    oof_preds[val_idx] = model.predict_proba(X_val_encoded)
    test_preds += model.predict_proba(X_test_encoded) / N_FOLDS
    
    # 计算当前折分数
    fold_score = balanced_accuracy_custom(y_val, oof_preds[val_idx])
    print(f"  Fold Score: {fold_score:.4f}")

# ========================
# 12. 计算CV分数
# ========================
cv_score = balanced_accuracy_custom(y, oof_preds)
print(f"\n" + "="*50)
print(f"CV分数: {cv_score:.6f}")
print("="*50)

# ========================
# 13. 生成提交文件
# ========================
final_preds = np.argmax(test_preds, axis=1)
final_labels = [idx2target[p] for p in final_preds]

submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': final_labels
})

submission.to_csv('submission_lgb_silver.csv', index=False)
print("\n提交文件已保存: submission_lgb_silver.csv")
print(f"\n预测分布:")
print(submission['Irrigation_Need'].value_counts())

print("\n完成！提交到Kaggle")

加载数据...
训练集: (630000, 21)
测试集: (270000, 20)
数值特征: 11
类别特征: 8

添加数字根特征...
删除单一值特征: 33

最终特征数: 152
其中类别特征: 74
训练特征形状: (630000, 86)
测试特征形状: (270000, 86)
类别权重: {np.int64(0): np.float64(0.5676949153458749), np.int64(1): np.float64(0.8783891180136694), np.int64(2): np.float64(9.995716121662145)}

训练 LightGBM

Fold 1/5
Training until validation scores don't improve for 200 rounds
[100]	valid_0's multi_logloss: 0.158469	valid_0's balanced_acc: 0.96145
[200]	valid_0's multi_logloss: 0.0961919	valid_0's balanced_acc: 0.966837
[300]	valid_0's multi_logloss: 0.0805538	valid_0's balanced_acc: 0.968906
[400]	valid_0's multi_logloss: 0.0737139	valid_0's balanced_acc: 0.97066
[500]	valid_0's multi_logloss: 0.0698863	valid_0's balanced_acc: 0.97165
[600]	valid_0's multi_logloss: 0.0671507	valid_0's balanced_acc: 0.97192
[700]	valid_0's multi_logloss: 0.0652609	valid_0's balanced_acc: 0.972818
[800]	valid_0's multi_logloss: 0.0636687	valid_0's balanced_acc: 0.973218
[900]	valid_0's multi_logloss: 0.0624

## version 5 = 0.93

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import optuna # 用于超参数优化
from optuna.samplers import TPESampler # 贝叶斯优化采样器
import warnings # 忽略警告
warnings.filterwarnings('ignore') # 这行代码会忽略所有的警告信息，保持输出清洁

# ========================
# 1. 设置随机种子
# ========================
SEED = 42 # 固定随机种子以确保结果可复现，随机种子的作用是确保每次运行代码时，生成的随机数序列相同，从而使得模型训练和评估的结果一致。这对于调试和比较不同模型或参数设置非常重要。
np.random.seed(SEED) # 设置NumPy的随机种子，确保所有使用NumPy生成的随机数都是可复现的

# ========================
# 2. 自定义评估指标
# ========================
def balanced_accuracy_custom(y_true, y_pred): # 定义一个函数来计算平衡准确率，这个函数将被用作LightGBM的自定义评估指标
    """计算平衡准确率 - 与竞赛评估一致"""
    if len(y_pred.shape) == 2:
        y_pred = np.argmax(y_pred, axis=1)
    
    n_classes = 3
    acc_per_class = 0
    for i in range(n_classes):
        mask = y_true == i
        if mask.sum() > 0:
            acc_per_class += np.sum((y_true[mask] == y_pred[mask])) / mask.sum()
    
    return acc_per_class / n_classes

def lgb_eval_metric(y_true, y_pred):
    """LightGBM自定义评估函数"""
    y_true = y_true.astype(int)
    y_pred = np.argmax(y_pred.reshape(-1, 3), axis=1)
    score = balanced_accuracy_custom(y_true, y_pred)
    return 'balanced_acc', score, True

# ========================
# 3. 加载数据
# ========================
print("="*60)
print("加载数据")
print("="*60)

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"训练集: {train.shape}")
print(f"测试集: {test.shape}")

target_col = 'Irrigation_Need'
drop_cols = ['id']

# 编码目标变量
target2idx = {v: i for i, v in enumerate(train[target_col].unique())}
idx2target = {v: i for i, v in target2idx.items()}
train[target_col] = train[target_col].map(target2idx)

# ========================
# 4. 识别特征类型
# ========================
CATS = [c for c in test.columns if train[c].dtype == object and c != target_col]
NUMS = [c for c in test.columns if c not in CATS and c not in drop_cols and c != target_col]

print(f"原始数值特征: {len(NUMS)}")
print(f"原始类别特征: {len(CATS)}")

# ========================
# 5. 扩展数字根特征（改进点1：从-4~4扩展到-6~6）
# ========================
def add_extended_digit_features(df, numeric_cols):
    """扩展数字根特征 - 捕捉更细粒度的模式"""
    df_new = df.copy()
    for c in numeric_cols:
        if c not in df_new.columns or c == 'id':
            continue
        # 扩展到 -6 到 6
        for k in range(-6, 7):
            try:
                df_new[f"{c}_digit{k}"] = (df_new[c] // (10**k) % 10).astype('int8')
            except:
                pass
        # 添加数字和特征
        try:
            df_new[f"{c}_digit_sum"] = df_new[c].astype(str).apply(
                lambda x: sum(int(d) for d in str(x).replace('.', '') if d.isdigit())
            ).astype('int16')
        except:
            pass
    return df_new

print("\n" + "="*60)
print("添加扩展数字根特征")
print("="*60)

train = add_extended_digit_features(train, NUMS)
test = add_extended_digit_features(test, NUMS)

# ========================
# 6. 添加交叉统计特征（改进点2）
# ========================
def add_cross_stats(df, numeric_cols, cat_cols):
    """添加类别与数值的交叉统计"""
    df_new = df.copy()
    
    # 选择最重要的特征（根据之前的特征重要性）
    important_cats = [c for c in cat_cols if c in df.columns][:3]
    important_nums = [c for c in numeric_cols if c in df.columns][:5]
    
    for cat in important_cats:
        for num in important_nums:
            try:
                # 每个类别下的数值统计
                group_stats = df_new.groupby(cat)[num].agg(['mean', 'std', 'max', 'min'])
                group_stats.columns = [f'{cat}_{num}_{col}' for col in ['mean', 'std', 'max', 'min']]
                # 使用merge避免索引问题
                df_new = df_new.reset_index(drop=True)
                group_stats = group_stats.reset_index()
                df_new = df_new.merge(group_stats, on=cat, how='left')
            except:
                pass
    
    return df_new

print("添加交叉统计特征...")
train = add_cross_stats(train, NUMS, CATS)
test = add_cross_stats(test, NUMS, CATS)

# ========================
# 7. 添加多项式特征（改进点3）
# ========================
def add_poly_features(df, numeric_cols):
    """添加多项式特征"""
    df_new = df.copy()
    
    # 最重要的数值特征
    important_nums = ['Soil_Moisture', 'Temperature_C', 'Rainfall_mm', 'Humidity']
    for col in important_nums:
        if col in df_new.columns:
            try:
                df_new[f'{col}_square'] = df_new[col] ** 2
                df_new[f'{col}_sqrt'] = np.sqrt(np.abs(df_new[col]) + 1e-8)
                df_new[f'{col}_log'] = np.log(np.abs(df_new[col]) + 1e-8)
            except:
                pass
    
    return df_new

print("添加多项式特征...")
train = add_poly_features(train, NUMS)
test = add_poly_features(test, NUMS)

# ========================
# 8. 删除单一值特征
# ========================
DROP = []
for c in test.columns:
    if test[c].nunique() == 1:
        DROP.append(c)

print(f"删除单一值特征: {len(DROP)}")
train = train.drop(DROP, axis=1)
test = test.drop(DROP, axis=1)

# ========================
# 9. 更新特征列表
# ========================
CATS = [c for c in test.columns if train[c].dtype == object and c != target_col]
NUMS = [c for c in test.columns if c not in CATS and c != target_col]

# 类别特征包括：原始类别 + 数字根特征
CATEGORY_FEATURES = CATS + [c for c in test.columns if 'digit' in c and c != target_col]

print(f"\n最终特征数: {len(NUMS) + len(CATEGORY_FEATURES)}")
print(f"数值特征: {len(NUMS)}")
print(f"类别特征: {len(CATEGORY_FEATURES)}")

# ========================
# 10. 准备训练数据
# ========================
X = train.drop([target_col], axis=1)
y = train[target_col].values
X_test = test.copy()

# 确保所有列都是数值类型
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)
for col in X_test.columns:
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(0)

print(f"训练特征形状: {X.shape}")
print(f"测试特征形状: {X_test.shape}")

# ========================
# 11. 计算样本权重
# ========================
unique, counts = np.unique(y, return_counts=True)
avg_count = len(y) / len(unique)
class_weights = {cls: avg_count / cnt for cls, cnt in zip(unique, counts)}
sample_weights = np.array([class_weights[y_i] for y_i in y])

print(f"类别权重: {class_weights}")

# ========================
# 12. LightGBM参数（银牌方案优化版）
# ========================
lgb_params = {
    "n_estimators": 6000,
    'boosting_type': 'gbdt',
    'max_depth': 4,
    'num_leaves': 32,
    'learning_rate': 0.05,
    'feature_fraction': 0.6,
    'bagging_fraction': 0.7,
    'bagging_freq': 1,
    'lambda_l1': 10,
    'lambda_l2': 10,
    'min_child_samples': 12,
    'random_state': SEED,
    'n_jobs': -1,
    'max_bin': 15000,
    'verbosity': -1,
    'subsample': 0.5,
    'subsample_for_bin': 100000,
    'subsample_freq': 1,
}

# ========================
# 13. 频率编码函数
# ========================
def apply_frequency_encoding(X_train, X_val, X_test, category_features):
    """应用频率编码"""
    X_train_enc = X_train.copy()
    X_val_enc = X_val.copy()
    X_test_enc = X_test.copy()
    
    for cat_col in category_features:
        if cat_col not in X_train.columns:
            continue
        
        # 计算频率映射
        freq_map = X_train[cat_col].value_counts(normalize=True).to_dict()
        
        # 应用编码
        X_train_enc[cat_col + '_freq'] = X_train[cat_col].map(freq_map).fillna(0)
        X_val_enc[cat_col + '_freq'] = X_val[cat_col].map(freq_map).fillna(0)
        X_test_enc[cat_col + '_freq'] = X_test[cat_col].map(freq_map).fillna(0)
        
        # 删除原始列
        X_train_enc = X_train_enc.drop(cat_col, axis=1)
        X_val_enc = X_val_enc.drop(cat_col, axis=1)
        X_test_enc = X_test_enc.drop(cat_col, axis=1)
    
    return X_train_enc, X_val_enc, X_test_enc

# ========================
# 14. 多种子训练（改进点4：多模型集成）
# ========================
print("\n" + "="*60)
print("多种子LightGBM训练")
print("="*60)

seeds = [42, 123, 456]
all_test_preds = []

for seed_idx, seed in enumerate(seeds):
    print(f"\n{'='*40}")
    print(f"训练模型 {seed_idx+1}/{len(seeds)} - 种子: {seed}")
    print(f"{'='*40}")
    
    # 更新随机种子
    lgb_params['random_state'] = seed
    np.random.seed(seed)
    
    N_FOLDS = 5
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    
    oof_preds = np.zeros((len(X), 3))
    test_preds = np.zeros((len(X_test), 3))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        print(f"Fold {fold+1}/{N_FOLDS}")
        
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        train_weights = sample_weights[train_idx]
        
        # 应用频率编码
        X_train_enc, X_val_enc, X_test_enc = apply_frequency_encoding(
            X_train, X_val, X_test, CATEGORY_FEATURES
        )
        
        # 确保所有列都是数值类型
        for col in X_train_enc.columns:
            X_train_enc[col] = pd.to_numeric(X_train_enc[col], errors='coerce').fillna(0)
            X_val_enc[col] = pd.to_numeric(X_val_enc[col], errors='coerce').fillna(0)
        for col in X_test_enc.columns:
            X_test_enc[col] = pd.to_numeric(X_test_enc[col], errors='coerce').fillna(0)
        
        # 训练模型
        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(
            X_train_enc, y_train,
            sample_weight=train_weights,
            eval_set=[(X_val_enc, y_val)],
            eval_metric=lgb_eval_metric,
            callbacks=[lgb.early_stopping(250), lgb.log_evaluation(100)]
        )
        
        # 预测
        oof_preds[val_idx] = model.predict_proba(X_val_enc)
        test_preds += model.predict_proba(X_test_enc) / N_FOLDS
        
        fold_score = balanced_accuracy_custom(y_val, oof_preds[val_idx])
        print(f"  Fold Score: {fold_score:.4f}, Best Iter: {model.best_iteration_}")
    
    # 计算CV分数
    cv_score = balanced_accuracy_custom(y, oof_preds)
    print(f"\n种子 {seed} CV分数: {cv_score:.6f}")
    
    all_test_preds.append(test_preds)

# ========================
# 15. 多模型集成（简单平均）
# ========================
print("\n" + "="*60)
print("模型集成")
print("="*60)

ensemble_test_preds = np.mean(all_test_preds, axis=0)

# ========================
# 16. 高级后处理优化（改进点5：Optuna优化）
# ========================
print("\n" + "="*60)
print("Optuna后处理优化")
print("="*60)

# 计算OOF分数用于优化
# 我们需要重新计算OOF预测（使用所有种子的平均）
all_oof_preds = []
for seed_idx, seed in enumerate(seeds):
    lgb_params['random_state'] = seed
    np.random.seed(seed)
    
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    oof_preds = np.zeros((len(X), 3))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        train_weights = sample_weights[train_idx]
        
        X_train_enc, X_val_enc, _ = apply_frequency_encoding(
            X_train, X_val, X_test, CATEGORY_FEATURES
        )
        
        for col in X_train_enc.columns:
            X_train_enc[col] = pd.to_numeric(X_train_enc[col], errors='coerce').fillna(0)
            X_val_enc[col] = pd.to_numeric(X_val_enc[col], errors='coerce').fillna(0)
        
        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(X_train_enc, y_train, sample_weight=train_weights)
        oof_preds[val_idx] = model.predict_proba(X_val_enc)
    
    all_oof_preds.append(oof_preds)

ensemble_oof = np.mean(all_oof_preds, axis=0)

def objective(trial):
    """Optuna优化目标函数"""
    # 类别权重
    w0 = trial.suggest_float('w0', 0.5, 3.0)
    w1 = trial.suggest_float('w1', 0.5, 3.0)
    w2 = trial.suggest_float('w2', 0.5, 3.0)
    
    # 温度参数（softmax温度）
    temp = trial.suggest_float('temp', 0.5, 2.0)
    
    # 应用权重和温度
    adjusted = ensemble_oof * np.array([w0, w1, w2])
    adjusted = np.exp(np.log(adjusted + 1e-8) / temp)
    adjusted = adjusted / adjusted.sum(axis=1, keepdims=True)
    
    return balanced_accuracy_custom(y, np.argmax(adjusted, axis=1))

# 运行优化
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED))
study.optimize(objective, n_trials=100, show_progress_bar=True)

print(f"\n最佳CV分数: {study.best_value:.6f}")
print(f"最佳参数:")
print(f"  w0: {study.best_params['w0']:.4f}")
print(f"  w1: {study.best_params['w1']:.4f}")
print(f"  w2: {study.best_params['w2']:.4f}")
print(f"  temp: {study.best_params['temp']:.4f}")

# 应用最佳参数到测试集
best_w = np.array([study.best_params['w0'], study.best_params['w1'], study.best_params['w2']])
best_temp = study.best_params['temp']

final_test_probs = ensemble_test_preds * best_w
final_test_probs = np.exp(np.log(final_test_probs + 1e-8) / best_temp)
final_test_probs = final_test_probs / final_test_probs.sum(axis=1, keepdims=True)

# ========================
# 17. 生成提交文件
# ========================
print("\n" + "="*60)
print("生成提交文件")
print("="*60)

final_preds = np.argmax(final_test_probs, axis=1)
final_labels = [idx2target[p] for p in final_preds]

submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': final_labels
})

submission.to_csv('submission_improved_silver.csv', index=False)

print(f"\n提交文件已保存: submission_improved_silver.csv")
print(f"\n预测分布:")
print(submission['Irrigation_Need'].value_counts())

# 计算最终CV分数
final_cv = balanced_accuracy_custom(y, np.argmax(ensemble_oof, axis=1))
print(f"\n最终CV分数: {final_cv:.6f}")

print("\n" + "="*60)
print("完成！提交 submission_improved_silver.csv 到 Kaggle")
print("预期分数: 0.980-0.983")
print("="*60)

加载数据
训练集: (630000, 21)
测试集: (270000, 20)
原始数值特征: 11
原始类别特征: 8

添加扩展数字根特征
添加交叉统计特征...
添加多项式特征...
删除单一值特征: 78

最终特征数: 267
数值特征: 160
类别特征: 107
训练特征形状: (630000, 168)
测试特征形状: (270000, 168)
类别权重: {np.int64(0): np.float64(0.5676949153458749), np.int64(1): np.float64(0.8783891180136694), np.int64(2): np.float64(9.995716121662145)}

多种子LightGBM训练

训练模型 1/3 - 种子: 42
Fold 1/5
Training until validation scores don't improve for 250 rounds
[100]	valid_0's multi_logloss: 0.249792	valid_0's balanced_acc: 0.912824
[200]	valid_0's multi_logloss: 0.225659	valid_0's balanced_acc: 0.916364
[300]	valid_0's multi_logloss: 0.215023	valid_0's balanced_acc: 0.919507
[400]	valid_0's multi_logloss: 0.208027	valid_0's balanced_acc: 0.921435
[500]	valid_0's multi_logloss: 0.203067	valid_0's balanced_acc: 0.92275
[600]	valid_0's multi_logloss: 0.19905	valid_0's balanced_acc: 0.924137
[700]	valid_0's multi_logloss: 0.195827	valid_0's balanced_acc: 0.924941
[800]	valid_0's multi_logloss: 0.193305	valid_0's balanced_ac

KeyboardInterrupt: 

## v6

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import balanced_accuracy_score
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# ========================
# 1. 设置随机种子
# ========================
SEED = 42
np.random.seed(SEED)

# ========================
# 2. 自定义评估指标
# ========================
def balanced_accuracy_custom(y_true, y_pred):
    """计算平衡准确率"""
    if len(y_pred.shape) == 2:
        y_pred = np.argmax(y_pred, axis=1)
    
    n_classes = 3
    acc_per_class = 0
    for i in range(n_classes):
        mask = y_true == i
        if mask.sum() > 0:
            acc_per_class += np.sum((y_true[mask] == y_pred[mask])) / mask.sum()
    
    return acc_per_class / n_classes

def lgb_eval_metric(y_true, y_pred):
    y_true = y_true.astype(int)
    y_pred = np.argmax(y_pred.reshape(-1, 3), axis=1)
    score = balanced_accuracy_custom(y_true, y_pred)
    return 'balanced_acc', score, True

# ========================
# 3. 加载数据
# ========================
print("="*50)
print("加载数据")
print("="*50)

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"训练集: {train.shape}")
print(f"测试集: {test.shape}")

target_col = 'Irrigation_Need'

# 编码目标变量
target2idx = {v: i for i, v in enumerate(train[target_col].unique())}
idx2target = {v: i for i, v in target2idx.items()}
train[target_col] = train[target_col].map(target2idx)

# ========================
# 4. 识别特征类型
# ========================
CATS = [c for c in test.columns if train[c].dtype == object and c != target_col]
NUMS = [c for c in test.columns if c not in CATS and c != target_col and c != 'id']

print(f"数值特征: {len(NUMS)}")
print(f"类别特征: {len(CATS)}")

# ========================
# 5. 数字根特征（银牌方案核心 - 保持原样）
# ========================
def add_digit_features(df, numeric_cols):
    df_new = df.copy()
    for c in numeric_cols:
        if c not in df_new.columns or c == 'id':
            continue
        for k in range(-4, 5):
            try:
                df_new[f"{c}_digit{k}"] = (df_new[c] // (10**k) % 10).astype('int8')
            except:
                pass
    return df_new

print("\n添加数字根特征...")
train = add_digit_features(train, NUMS)
test = add_digit_features(test, NUMS)

# ========================
# 6. 删除单一值特征
# ========================
DROP = []
for c in test.columns:
    if test[c].nunique() == 1:
        DROP.append(c)

print(f"删除单一值特征: {len(DROP)}")
train = train.drop(DROP, axis=1)
test = test.drop(DROP, axis=1)

# ========================
# 7. 更新特征列表
# ========================
CATS = [c for c in test.columns if train[c].dtype == object and c != target_col]
CATEGORY_FEATURES = CATS + [c for c in test.columns if 'digit' in c]

print(f"最终特征数: {len(train.columns) - 2}")  # 减2: id和target
print(f"类别特征数: {len(CATEGORY_FEATURES)}")

# ========================
# 8. 准备训练数据
# ========================
X = train.drop([target_col, 'id'], axis=1)
y = train[target_col].values
X_test = test.drop(['id'], axis=1)

# 确保数值类型
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)
for col in X_test.columns:
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(0)

print(f"训练特征形状: {X.shape}")
print(f"测试特征形状: {X_test.shape}")

# ========================
# 9. 计算样本权重
# ========================
unique, counts = np.unique(y, return_counts=True)
avg_count = len(y) / len(unique)
sample_weights = np.array([avg_count / counts[y_i] for y_i in y])

# ========================
# 10. 频率编码函数
# ========================
def apply_frequency_encoding(X_train, X_val, X_test, category_features):
    X_train_enc = X_train.copy()
    X_val_enc = X_val.copy()
    X_test_enc = X_test.copy()
    
    for cat_col in category_features:
        if cat_col not in X_train.columns:
            continue
        
        freq_map = X_train[cat_col].value_counts(normalize=True).to_dict()
        
        X_train_enc[cat_col + '_freq'] = X_train[cat_col].map(freq_map).fillna(0)
        X_val_enc[cat_col + '_freq'] = X_val[cat_col].map(freq_map).fillna(0)
        X_test_enc[cat_col + '_freq'] = X_test[cat_col].map(freq_map).fillna(0)
        
        X_train_enc = X_train_enc.drop(cat_col, axis=1)
        X_val_enc = X_val_enc.drop(cat_col, axis=1)
        X_test_enc = X_test_enc.drop(cat_col, axis=1)
    
    return X_train_enc, X_val_enc, X_test_enc

# ========================
# 11. LightGBM参数（微调版 - 基于0.97305优化）
# ========================
lgb_params = {
    'n_estimators': 8000,        # 从6000增加到8000
    'boosting_type': 'gbdt',
    'max_depth': 4,
    'num_leaves': 31,             # 从32微调
    'learning_rate': 0.03,        # 从0.05降低
    'feature_fraction': 0.6,
    'bagging_fraction': 0.7,
    'bagging_freq': 1,
    'lambda_l1': 8,               # 从10降低
    'lambda_l2': 8,               # 从10降低
    'min_child_samples': 12,
    'random_state': SEED,
    'n_jobs': -1,
    'max_bin': 15000,
    'verbosity': -1,
    'subsample': 0.5,
    'subsample_for_bin': 100000,
    'subsample_freq': 1,
}

# ========================
# 12. XGBoost参数（新增）
# ========================
xgb_params = {
    'n_estimators': 3000,
    'max_depth': 5,
    'learning_rate': 0.03,
    'subsample': 0.7,
    'colsample_bytree': 0.6,
    'min_child_weight': 5,
    'gamma': 0.1,
    'reg_alpha': 1.0,
    'reg_lambda': 2.0,
    'objective': 'multi:softprob',
    'num_class': 3,
    'eval_metric': 'mlogloss',
    'random_state': SEED,
    'n_jobs': -1,
    'verbosity': 0
}

# ========================
# 13. 训练LightGBM
# ========================
print("\n" + "="*50)
print("训练 LightGBM")
print("="*50)

N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

lgb_oof = np.zeros((len(X), 3))
lgb_test = np.zeros((len(X_test), 3))

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\nFold {fold+1}/{N_FOLDS}")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    train_weights = sample_weights[train_idx]
    
    # 频率编码
    X_train_enc, X_val_enc, X_test_enc = apply_frequency_encoding(
        X_train, X_val, X_test, CATEGORY_FEATURES
    )
    
    # 确保数值类型
    for col in X_train_enc.columns:
        X_train_enc[col] = pd.to_numeric(X_train_enc[col], errors='coerce').fillna(0)
        X_val_enc[col] = pd.to_numeric(X_val_enc[col], errors='coerce').fillna(0)
    for col in X_test_enc.columns:
        X_test_enc[col] = pd.to_numeric(X_test_enc[col], errors='coerce').fillna(0)
    
    # 训练
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_train_enc, y_train,
        sample_weight=train_weights,
        eval_set=[(X_val_enc, y_val)],
        eval_metric=lgb_eval_metric,
        callbacks=[lgb.early_stopping(200), lgb.log_evaluation(100)]
    )
    
    lgb_oof[val_idx] = model.predict_proba(X_val_enc)
    lgb_test += model.predict_proba(X_test_enc) / N_FOLDS
    
    fold_score = balanced_accuracy_custom(y_val, lgb_oof[val_idx])
    print(f"  Score: {fold_score:.4f}, Best iter: {model.best_iteration_}")

lgb_cv = balanced_accuracy_custom(y, lgb_oof)
print(f"\nLightGBM CV: {lgb_cv:.6f}")

# ========================
# 14. 训练XGBoost（可选，如果时间不够可以跳过）
# ========================
print("\n" + "="*50)
print("训练 XGBoost")
print("="*50)

xgb_oof = np.zeros((len(X), 3))
xgb_test = np.zeros((len(X_test), 3))

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\nFold {fold+1}/{N_FOLDS}")
    
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    train_weights = sample_weights[train_idx]
    
    # 频率编码
    X_train_enc, X_val_enc, X_test_enc = apply_frequency_encoding(
        X_train, X_val, X_test, CATEGORY_FEATURES
    )
    
    # 确保数值类型
    for col in X_train_enc.columns:
        X_train_enc[col] = pd.to_numeric(X_train_enc[col], errors='coerce').fillna(0)
        X_val_enc[col] = pd.to_numeric(X_val_enc[col], errors='coerce').fillna(0)
    for col in X_test_enc.columns:
        X_test_enc[col] = pd.to_numeric(X_test_enc[col], errors='coerce').fillna(0)
    
    # 训练（XGBoost需要设置sample_weight）
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_train_enc, y_train, sample_weight=train_weights)
    
    xgb_oof[val_idx] = model.predict_proba(X_val_enc)
    xgb_test += model.predict_proba(X_test_enc) / N_FOLDS
    
    fold_score = balanced_accuracy_custom(y_val, xgb_oof[val_idx])
    print(f"  Score: {fold_score:.4f}")

xgb_cv = balanced_accuracy_custom(y, xgb_oof)
print(f"\nXGBoost CV: {xgb_cv:.6f}")

# ========================
# 15. 模型集成（简单平均）
# ========================
print("\n" + "="*50)
print("模型集成")
print("="*50)

# 加权平均（根据CV分数）
total_cv = lgb_cv + xgb_cv
lgb_weight = lgb_cv / total_cv
xgb_weight = xgb_cv / total_cv

print(f"LightGBM 权重: {lgb_weight:.3f}")
print(f"XGBoost 权重: {xgb_weight:.3f}")

ensemble_test = lgb_weight * lgb_test + xgb_weight * xgb_test
ensemble_cv = balanced_accuracy_custom(y, 
    lgb_weight * np.argmax(lgb_oof, axis=1) + 
    xgb_weight * np.argmax(xgb_oof, axis=1)
)
print(f"集成CV分数: {ensemble_cv:.6f}")

# ========================
# 16. 生成提交文件
# ========================
print("\n" + "="*50)
print("生成提交文件")
print("="*50)

final_preds = np.argmax(ensemble_test, axis=1)
final_labels = [idx2target[p] for p in final_preds]

submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': final_labels
})

submission.to_csv('submission_final_optimized.csv', index=False)

print(f"提交文件: submission_final_optimized.csv")
print(f"\n预测分布:")
print(submission['Irrigation_Need'].value_counts())

print("\n" + "="*50)
print("完成！提交到 Kaggle")
print(f"预期分数: 0.974-0.976")
print("="*50)

加载数据
训练集: (630000, 21)
测试集: (270000, 20)
数值特征: 11
类别特征: 8

添加数字根特征...
删除单一值特征: 33
最终特征数: 85
类别特征数: 74
训练特征形状: (630000, 85)
测试特征形状: (270000, 85)

训练 LightGBM

Fold 1/5
Training until validation scores don't improve for 200 rounds
[100]	valid_0's multi_logloss: 0.56587	valid_0's balanced_acc: 0.79628
[200]	valid_0's multi_logloss: 0.513826	valid_0's balanced_acc: 0.801114
[300]	valid_0's multi_logloss: 0.490887	valid_0's balanced_acc: 0.809295
[400]	valid_0's multi_logloss: 0.477345	valid_0's balanced_acc: 0.815359
[500]	valid_0's multi_logloss: 0.466554	valid_0's balanced_acc: 0.820622
[600]	valid_0's multi_logloss: 0.457799	valid_0's balanced_acc: 0.824694
[700]	valid_0's multi_logloss: 0.450458	valid_0's balanced_acc: 0.828447
[800]	valid_0's multi_logloss: 0.444605	valid_0's balanced_acc: 0.830932
[900]	valid_0's multi_logloss: 0.438883	valid_0's balanced_acc: 0.833752
[1000]	valid_0's multi_logloss: 0.433711	valid_0's balanced_acc: 0.83646
[1100]	valid_0's multi_logloss: 0.42942	val

## v7=0.978

In [ ]:
# Imports & Configuration
import gc
import warnings
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from scipy.optimize import minimize
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, TargetEncoder
from sklearn.utils.class_weight import compute_sample_weight
from tqdm import tqdm

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
print("✅ Imports complete")

class Config:
    TRAIN_PATH    = "train.csv"
    TEST_PATH     = "test.csv"
    ORIGINAL_PATH = "sample_submission.csv"

    TARGET         = "Irrigation_Need"
    TARGET_MAPPING = {"Low": 0, "Medium": 1, "High": 2}
    INV_MAPPING    = {0: "Low", 1: "Medium", 2: "High"}

    N_FOLDS       = 10
    RANDOM_SEED   = 42
    PSEUDO_THRESH = 0.92

    SOIL_THRESH = 25
    RAIN_THRESH = 300
    TEMP_THRESH = 30
    WIND_THRESH = 10

    CAT_COLS = [
        "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
        "Irrigation_Type", "Water_Source", "Mulching_Used", "Region",
    ]
    NUM_COLS = [
        "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
        "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
        "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm",
    ]

    LOGIT_COEFS = {
        "Low": {
            "intercept": 16.3173, "soil_lt_25": -11.0237, "temp_gt_30": -5.8559,
            "rain_lt_300": -10.8500, "wind_gt_10": -5.8284,
            "Flowering": -5.4155, "Harvest": 5.5073, "Sowing": 5.2299, "Vegetative": -5.4617,
            "Mulch_No": -3.0014, "Mulch_Yes": 2.8613,
        },
        "Medium": {
            "intercept": 4.6524, "soil_lt_25": 0.3290, "temp_gt_30": -0.0204,
            "rain_lt_300": 0.1542, "wind_gt_10": 0.0841,
            "Flowering": 0.3586, "Harvest": -0.1348, "Sowing": -0.3547, "Vegetative": 0.3334,
            "Mulch_No": 0.1883, "Mulch_Yes": 0.0142,
        },
        "High": {
            "intercept": -20.9697, "soil_lt_25": 10.6947, "temp_gt_30": 5.8763,
            "rain_lt_300": 10.6958, "wind_gt_10": 5.7444,
            "Flowering": 5.0569, "Harvest": -5.3725, "Sowing": -4.8752, "Vegetative": 5.1283,
            "Mulch_No": 2.8131, "Mulch_Yes": -2.8755,
        },
    }

    XGB_PARAMS = dict(
        max_depth=4,
        learning_rate=0.0305,
        min_child_weight=2.334,
        subsample=0.9766,
        colsample_bytree=0.5353,
        gamma=4.2585,
        reg_alpha=4.0829e-08,
        reg_lambda=0.000135,
        objective="multi:softprob",
        num_class=3,
        tree_method="hist",
        enable_categorical=True,
        eval_metric="mlogloss",
        seed=42,
    )

cfg = Config()
print("✅ Config ready")

# Data Loading
train_raw = pd.read_csv(cfg.TRAIN_PATH).dropna(subset=[cfg.TARGET])
test      = pd.read_csv(cfg.TEST_PATH)
original  = pd.read_csv(cfg.ORIGINAL_PATH).rename(columns={"Irrigation_Requirement": cfg.TARGET})

original["id"] = range(train_raw["id"].max() + 1, train_raw["id"].max() + 1 + len(original))
train     = pd.concat([train_raw, original], ignore_index=True)
n_comp    = len(train_raw)

print(f"Competition train : {len(train_raw):>7,}")
print(f"Original dataset  : {len(original):>7,}")
print(f"Combined train    : {len(train):>7,}")
print(f"Test              : {len(test):>7,}")

# ==========================
# 🔥 高分核心：超强特征工程
# ==========================
def add_binary_flags(df):
    df["soil_lt_25"]   = (df["Soil_Moisture"]      < cfg.SOIL_THRESH).astype(int)
    df["rain_lt_300"]  = (df["Rainfall_mm"]         < cfg.RAIN_THRESH).astype(int)
    df["temp_gt_30"]   = (df["Temperature_C"]       > cfg.TEMP_THRESH).astype(int)
    df["wind_gt_10"]   = (df["Wind_Speed_kmh"]      > cfg.WIND_THRESH).astype(int)
    df["is_harvest"]   = (df["Crop_Growth_Stage"]  == "Harvest").astype(int)
    df["is_sowing"]    = (df["Crop_Growth_Stage"]  == "Sowing").astype(int)
    df["mulching_yes"] = (df["Mulching_Used"]       == "Yes").astype(int)
    return df

def add_magic_score(df):
    high = 2 * df["soil_lt_25"] + 2 * df["rain_lt_300"] + df["temp_gt_30"] + df["wind_gt_10"]
    low  = 2 * df["is_harvest"] + 2 * df["is_sowing"] + df["mulching_yes"]
    df["magic_score"] = high - low
    df["dist_boundary_0"] = (df["magic_score"] - 0).abs()
    df["dist_boundary_3"] = (df["magic_score"] - 3).abs()
    return df

def add_decimal_digits(df):
    cols = ["Soil_Moisture", "Temperature_C", "Rainfall_mm", "Wind_Speed_kmh",
            "Humidity", "Soil_pH", "Organic_Carbon", "Electrical_Conductivity",
            "Sunlight_Hours", "Field_Area_hectare", "Previous_Irrigation_mm"]
    for col in cols:
        v = df[col].values
        df[f"{col}_dec"] = np.floor((v - np.floor(v)) * 10).astype(int)
    return df

def add_threshold_distances(df):
    df["soil_dist_25"]  = df["Soil_Moisture"]  - cfg.SOIL_THRESH
    df["rain_dist_300"] = df["Rainfall_mm"]   - cfg.RAIN_THRESH
    df["temp_dist_30"]  = df["Temperature_C"] - cfg.TEMP_THRESH
    df["wind_dist_10"]  = df["Wind_Speed_kmh"] - cfg.WIND_THRESH
    return df

def add_logit_scores(df):
    flags = {
        "Flowering":  (df["Crop_Growth_Stage"] == "Flowering").astype(float),
        "Harvest":    (df["Crop_Growth_Stage"] == "Harvest").astype(float),
        "Sowing":     (df["Crop_Growth_Stage"] == "Sowing").astype(float),
        "Vegetative": (df["Crop_Growth_Stage"] == "Vegetative").astype(float),
        "Mulch_No":   (df["Mulching_Used"]     == "No").astype(float),
        "Mulch_Yes":  (df["Mulching_Used"]     == "Yes").astype(float),
    }
    binary = {
        "soil_lt_25": df["soil_lt_25"].astype(float),
        "temp_gt_30": df["temp_gt_30"].astype(float),
        "rain_lt_300": df["rain_lt_300"].astype(float),
        "wind_gt_10": df["wind_gt_10"].astype(float),
    }
    for cls, coefs in cfg.LOGIT_COEFS.items():
        df[f"logit_{cls}"] = coefs["intercept"]
        for k, v in binary.items(): df[f"logit_{cls}"] += coefs[k] * v
        for k, v in flags.items(): df[f"logit_{cls}"] += coefs[k] * v
    return df

def add_domain_features(df):
    rain, prev, temp, sun = df["Rainfall_mm"], df["Previous_Irrigation_mm"], df["Temperature_C"], df["Sunlight_Hours"]
    humid, wind, moist    = df["Humidity"], df["Wind_Speed_kmh"], df["Soil_Moisture"]
    oc, ec, area, ph      = df["Organic_Carbon"], df["Electrical_Conductivity"], df["Field_Area_hectare"], df["Soil_pH"]
    mulch                 = df["Mulching_Used"].map({"Yes":1, "No":0}).fillna(0)

    et_proxy    = (temp * sun) / (humid + 1)
    total_water = rain + prev

    df["Total_Water_Input"] = total_water
    df["Moisture_Deficit"]  = 100 - moist
    df["Irrigation_Ratio"]  = prev / (rain + 1)
    df["ET_Proxy"]          = et_proxy
    df["Evap_Stress"]       = (temp * wind) / (humid + 1)
    df["Net_Water_Need"]    = et_proxy - rain/10
    df["VPD_Proxy"]         = temp * (1 - humid/100)
    df["Heat_Stress"]       = temp * (100-humid)/100
    df["Dryness_Index"]     = temp * sun / (rain + 1)
    df["Drought_Risk"]      = df["Dryness_Index"] * df["Moisture_Deficit"] / 100
    return df

def engineer_features(df):
    df = df.copy()
    df = add_binary_flags(df)
    df = add_magic_score(df)
    # df = add_decimal_digits(df)
    df = add_threshold_distances(df)
    df = add_logit_scores(df)
    df = add_domain_features(df)
    return df

# 特征工程
train_eng = engineer_features(train)
test_eng  = engineer_features(test)
print(f"Train shape: {train_eng.shape}, Test shape: {test_eng.shape}")

# 类别编码（无泄露版）
for col in cfg.CAT_COLS:
    le = LabelEncoder()
    le.fit(pd.concat([train_eng[col].astype(str), test_eng[col].astype(str)]))
    train_eng[col] = le.transform(train_eng[col].astype(str)).astype("int32")
    test_eng[col]  = le.transform(test_eng[col].astype(str)).astype("int32")

# 交互特征
n_train = len(train_eng)
interaction_cols = []
for c1, c2 in tqdm(combinations(cfg.NUM_COLS + cfg.CAT_COLS, 2), desc="Interactions"):
    name = f"{c1}|{c2}"
    combined = pd.concat([
        train_eng[c1].astype(str)+"_"+train_eng[c2].astype(str),
        test_eng[c1].astype(str)+"_"+test_eng[c2].astype(str)
    ])
    codes, _ = combined.factorize()
    if pd.Series(codes).nunique() > len(codes)//2: continue
    train_eng[name] = codes[:n_train]
    test_eng[name]  = codes[n_train:]
    interaction_cols.append(name)

print(f"Created {len(interaction_cols)} interaction features")

# 构建特征矩阵
y = train_eng[cfg.TARGET].map(cfg.TARGET_MAPPING).values
drop_cols = {"id", cfg.TARGET, *interaction_cols}
base_feats = [c for c in train_eng.columns if c not in drop_cols and train_eng[c].dtype!=object]
med = train_eng[base_feats].median()

X_base  = train_eng[base_feats].fillna(med).astype("float32")
X_tbase = test_eng[base_feats].fillna(med).astype("float32")
X_pair  = train_eng[interaction_cols]
X_tpair = test_eng[interaction_cols]

# ==========================
# 🔥 10折CV + 目标编码
# ==========================
def apply_target_encoding(X_tr_p, y_tr, X_va_p, X_te_p, cols):
    enc = TargetEncoder(target_type="multiclass", cv=5, random_state=cfg.RANDOM_SEED)
    tr_enc = pd.DataFrame(enc.fit_transform(X_tr_p[cols], y_tr))
    va_enc = pd.DataFrame(enc.transform(X_va_p[cols]))
    te_enc = pd.DataFrame(enc.transform(X_te_p[cols]))
    return tr_enc, va_enc, te_enc

def balanced_accuracy_metric(preds, dmatrix):
    labels = dmatrix.get_label().astype(int)
    y_pred = preds.reshape(-1,3).argmax(axis=1)
    return "bal_ACC", balanced_accuracy_score(labels, y_pred)

skf = StratifiedKFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.RANDOM_SEED)
oof_probs  = np.zeros((len(X_base), 3))
test_probs = np.zeros((len(X_tbase), 3))
best_iters = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_base, y)):
    print(f"\nFold {fold+1}/{cfg.N_FOLDS}")
    tr_enc, va_enc, te_enc = apply_target_encoding(
        X_pair.iloc[tr_idx], y[tr_idx], X_pair.iloc[val_idx], X_tpair, interaction_cols
    )
    Xt = pd.concat([X_base.iloc[tr_idx].reset_index(drop=True), tr_enc], axis=1)
    Xv = pd.concat([X_base.iloc[val_idx].reset_index(drop=True), va_enc], axis=1)
    Xe = pd.concat([X_tbase.reset_index(drop=True), te_enc], axis=1)

    dtrain = xgb.DMatrix(Xt, label=y[tr_idx])
    dval   = xgb.DMatrix(Xv, label=y[val_idx])
    dtest  = xgb.DMatrix(Xe)

    model = xgb.train(
        cfg.XGB_PARAMS, dtrain, 10000,
        evals=[(dtrain,"train"),(dval,"val")],
        custom_metric=balanced_accuracy_metric, maximize=True,
        early_stopping_rounds=300, verbose_eval=500
    )
    oof_probs[val_idx] = model.predict(dval)
    test_probs += model.predict(dtest) / cfg.N_FOLDS
    best_iters.append(model.best_iteration)
    gc.collect()

baseline_ba = balanced_accuracy_score(y, oof_probs.argmax(axis=1))
print(f"\nOOF Balanced Accuracy: {baseline_ba:.5f}")

# ==========================
# 🔥 伪标签（高分关键）
# ==========================
mask = test_probs.max(axis=1) >= cfg.PSEUDO_THRESH
print(f"Pseudo-label samples: {mask.sum()}")

if mask.sum() >= 500:
    tr_enc_full, _, te_enc_full = apply_target_encoding(X_pair, y, X_pair, X_tpair, interaction_cols)
    X_full   = pd.concat([X_base, tr_enc_full], axis=1)
    X_test_f = pd.concat([X_tbase, te_enc_full], axis=1)
    X_aug = pd.concat([X_full, X_test_f[mask]], ignore_index=True)
    y_aug = np.concatenate([y, test_probs[mask].argmax(axis=1)])

    final_model = xgb.XGBClassifier(**cfg.XGB_PARAMS, n_estimators=int(np.mean(best_iters)))
    final_model.fit(X_aug, y_aug, verbose=False)
    final_test_probs = final_model.predict_proba(X_test_f)
else:
    final_test_probs = test_probs

# ==========================
# 🔥 类别权重优化（直接涨0.01+）
# ==========================
def neg_bal_acc(w, probs, y):
    return -balanced_accuracy_score(y, (probs * w).argmax(axis=1))

best_w = [0.9, 1.0, 5.0]
res = minimize(neg_bal_acc, x0=best_w, args=(oof_probs, y), method="Nelder-Mead")
final_w = res.x if -res.fun > baseline_ba else np.array(best_w)
opt_ba = balanced_accuracy_score(y, (oof_probs * final_w).argmax(axis=1))

print(f"Optimized BA: {opt_ba:.5f}")
print(f"Weight: {np.round(final_w, 4)}")

# 提交
test_preds = (final_test_probs * final_w).argmax(axis=1)
sub = pd.DataFrame({
    "id": test["id"],
    cfg.TARGET: [cfg.INV_MAPPING[p] for p in test_preds]
})
sub.to_csv("submission.csv", index=False)
print("✅ 提交文件已生成！")
print(sub[cfg.TARGET].value_counts())

✅ Imports complete
✅ Config ready
Competition train : 630,000
Original dataset  : 270,000
Combined train    : 900,000
Test              : 270,000
Train shape: (900000, 59), Test shape: (270000, 58)


Interactions: 171it [01:17,  2.20it/s]


Created 144 interaction features

Fold 1/10
[0]	train-mlogloss:0.64000	train-bal_ACC:0.33333	val-mlogloss:0.63990	val-bal_ACC:0.33333
[500]	train-mlogloss:0.03052	train-bal_ACC:0.97411	val-mlogloss:0.03163	val-bal_ACC:0.97285
[1000]	train-mlogloss:0.02820	train-bal_ACC:0.97625	val-mlogloss:0.03103	val-bal_ACC:0.97348
[1016]	train-mlogloss:0.02819	train-bal_ACC:0.97624	val-mlogloss:0.03102	val-bal_ACC:0.97348

Fold 2/10
[0]	train-mlogloss:0.63999	train-bal_ACC:0.33333	val-mlogloss:0.64001	val-bal_ACC:0.33333
[500]	train-mlogloss:0.03040	train-bal_ACC:0.97414	val-mlogloss:0.03367	val-bal_ACC:0.97135
[981]	train-mlogloss:0.02808	train-bal_ACC:0.97630	val-mlogloss:0.03303	val-bal_ACC:0.97202

Fold 3/10
[0]	train-mlogloss:0.63999	train-bal_ACC:0.33333	val-mlogloss:0.64001	val-bal_ACC:0.33333
[500]	train-mlogloss:0.03052	train-bal_ACC:0.97391	val-mlogloss:0.03302	val-bal_ACC:0.97130
[1000]	train-mlogloss:0.02817	train-bal_ACC:0.97615	val-mlogloss:0.03241	val-bal_ACC:0.97173
[1068]	train-mlog

## v8=0.968

In [ ]:
# ============================================
# 🔥 灌溉需求预测 - 优化完整版
# 整合多模型集成、高阶特征、智能伪标签
# ============================================

# Imports & Configuration
import gc
import warnings
from itertools import combinations
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from scipy.optimize import minimize
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from imblearn.over_sampling import SMOTE
from tqdm import tqdm

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
print("✅ Imports complete")

# ============================================
# 配置类 - 优化版，这个模块包含了所有路径、参数和超参数设置，方便统一管理和调整，是整个代码的核心配置部分。面试的时候可以重点介绍这个类的设计思路和参数选择依据。一共有100多行，涵盖了数据路径、目标映射、交叉验证设置、特征工程参数、模型参数等，是整个项目的基础配置。通过这个类，我们可以轻松调整模型和特征工程的细节，而不需要修改其他部分的代码，提高了代码的可维护性和灵活性。
# ============================================
class Config:#config的意思是配置，这个类主要用来存储和管理整个项目的配置参数，包括数据路径、目标映射、交叉验证设置、特征工程参数、模型参数等。通过这个类，我们可以集中管理所有的配置项，方便在需要调整参数时只修改这个类，而不需要在代码的其他部分进行修改，提高了代码的可维护性和灵活性。
    TRAIN_PATH    = "train.csv"
    TEST_PATH     = "test.csv"
    ORIGINAL_PATH = "sample_submission.csv"

    TARGET         = "Irrigation_Need"
    TARGET_MAPPING = {"Low": 0, "Medium": 1, "High": 2}
    INV_MAPPING    = {0: "Low", 1: "Medium", 2: "High"}

    N_FOLDS       = 10 # 交叉验证的折数，设置为10表示我们将数据分成10份，每次使用其中9份进行训练，1份进行验证，这样可以更稳定地评估模型的性能，同时也能充分利用数据进行训练。
    RANDOM_SEED   = 42 # 随机种子，设置为42是因为它是一个常用的随机数种子，可以确保结果的可复现性，同时也具有一定的象征意义（被称为“生命、宇宙以及一切的答案”），在机器学习中使用固定的随机种子可以帮助我们在调试和比较模型时获得一致的结果。
    N_PSEUDO_ITER = 3  # 伪标签迭代次数

    SOIL_THRESH = 25 # 土壤湿度阈值，设置为25表示当土壤湿度低于25时，可能需要灌溉，这个阈值可以根据领域知识和数据分布进行调整，以更好地捕捉灌溉需求的模式。
    RAIN_THRESH = 300 # 降雨量阈值，设置为300表示当降雨量低于300毫米时，可能需要灌溉，这个阈值可以根据历史数据和作物需求进行调整，以更准确地预测灌溉需求。
    TEMP_THRESH = 30 # 温度阈值，设置为30表示当温度高于30°C时，可能需要灌溉，这个阈值可以根据作物的生长特性和气候条件进行调整。
    WIND_THRESH = 10 # 风速阈值，设置为10表示当风速高于10 km/h时，可能需要灌溉，这个阈值可以根据作物的抗风能力和气候条件进行调整。

    CAT_COLS = [
        "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
        "Irrigation_Type", "Water_Source", "Mulching_Used", "Region",
    ] # 类别特征列表，包含了土壤类型、作物类型、作物生长阶段、季节、灌溉类型、水源、是否使用覆盖物和地区等特征，这些特征在模型训练中会进行编码处理。
    NUM_COLS = [
        "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
        "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
        "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm",
    ] # 数值特征列表，包含了土壤pH值、土壤湿度、有机碳、电导率、温度、湿度、降雨量、日照小时数、风速、田地面积和之前的灌溉量等特征，这些特征在模型训练中会进行数值处理和特征工程。

    LOGIT_COEFS = {
        "Low": {
            "intercept": 16.3173, "soil_lt_25": -11.0237, "temp_gt_30": -5.8559,
            "rain_lt_300": -10.8500, "wind_gt_10": -5.8284,
            "Flowering": -5.4155, "Harvest": 5.5073, "Sowing": 5.2299, "Vegetative": -5.4617,
            "Mulch_No": -3.0014, "Mulch_Yes": 2.8613,
        }, # 这个字典存储了针对每个类别（Low、Medium、High）的逻辑回归系数，这些系数是根据特征对目标变量的影响程度计算得出的，可以用来构建一个简单的逻辑回归模型，作为特征工程的一部分，帮助模型更好地捕捉特征与目标之间的关系。
        "Medium": {
            "intercept": 4.6524, "soil_lt_25": 0.3290, "temp_gt_30": -0.0204, # 这行是针对Medium类别的逻辑回归系数，表示当特征soil_lt_25为1时，对Medium类别的logit值增加0.3290，当temp_gt_30为1时，对Medium类别的logit值减少0.0204，以此类推，这些系数可以用来计算每个样本属于Medium类别的概率。
            "rain_lt_300": 0.1542, "wind_gt_10": 0.0841, # 这行继续列出了针对Medium类别的其他特征的逻辑回归系数，表示当rain_lt_300为1时，对Medium类别的logit值增加0.1542，当wind_gt_10为1时，对Medium类别的logit值增加0.0841，以此类推，这些系数可以用来计算每个样本属于Medium类别的概率。
            "Flowering": 0.3586, "Harvest": -0.1348, "Sowing": -0.3547, "Vegetative": 0.3334,
            "Mulch_No": 0.1883, "Mulch_Yes": 0.0142,
        },
        "High": {
            "intercept": -20.9697, "soil_lt_25": 10.6947, "temp_gt_30": 5.8763,
            "rain_lt_300": 10.6958, "wind_gt_10": 5.7444,
            "Flowering": 5.0569, "Harvest": -5.3725, "Sowing": -4.8752, "Vegetative": 5.1283,
            "Mulch_No": 2.8131, "Mulch_Yes": -2.8755,
        },
    }

    # XGBoost参数优化
    XGB_PARAMS = dict(
        max_depth=5, # 这个参数是XGBoost模型的最大树深度，设置为5可以让模型捕捉到更复杂的特征交互关系，但同时也增加了过拟合的风险，需要通过交叉验证来确定最优值。
        learning_rate=0.025,# 这个是学习率，设置为0.025可以让模型在每次迭代中更稳健地更新权重，避免过快地收敛到局部最优解，同时也需要更多的迭代次数来达到最佳性能。
        min_child_weight=3, # 这个参数是XGBoost模型的最小子样本权重和，设置为3可以让模型在分裂节点时更谨慎，避免过拟合，同时也可能导致欠拟合，需要通过交叉验证来确定最优值。
        subsample=0.85,# 这个参数是XGBoost模型的子样本采样率，设置为0.85可以让模型在每次迭代中随机采样85%的训练数据，增加模型的泛化能力，减少过拟合的风险。
        colsample_bytree=0.6,# 这个参数是XGBoost模型的列采样率，设置为0.6可以让模型在每次迭代中随机采样60%的特征，增加模型的泛化能力，减少过拟合的风险，同时也可以提高训练速度。
        gamma=2.0,# 这个参数是XGBoost模型的最小分裂损失，设置为2.0可以让模型在分裂节点时更谨慎，只有当分裂后的损失减少超过2.0时才会进行分裂，这可以帮助模型避免过拟合，但也可能导致欠拟合，需要通过交叉验证来确定最优值。
        reg_alpha=0.01,# 这个参数是XGBoost模型的L1正则化项系数，设置为0.01可以让模型在训练过程中对特征权重进行稀疏化处理，帮助模型选择重要特征，减少过拟合的风险。
        reg_lambda=1.0, # 这个参数是XGBoost模型的L2正则化项系数，设置为1.0可以让模型在训练过程中对特征权重进行平滑处理，帮助模型减少过拟合的风险，同时也可以提高模型的稳定性。
        objective="multi:softprob", # 这个参数是XGBoost模型的目标函数，设置为"multi:softprob"表示这是一个多分类问题，模型会输出每个类别的概率分布，而不是直接输出类别标签，这对于评估模型性能和进行后续处理非常有用。
        num_class=3, # 这个参数是XGBoost模型的类别数量，设置为3表示目标变量有三个类别（Low、Medium、High），模型会根据这个参数来调整输出的概率分布维度。
        tree_method="hist", # 这个参数是XGBoost模型的树构建方法，设置为"hist"表示使用基于直方图的算法来构建树，这种方法在处理大规模数据时更高效，可以加快训练速度，同时也可以减少内存使用。
        enable_categorical=True, # 这个参数是XGBoost模型的类别特征处理选项，设置为True表示模型会自动识别和处理类别特征，无需手动进行独热编码或标签编码，这可以简化数据预处理流程，同时也可以提高模型的性能。
        eval_metric="mlogloss", # 这个参数是XGBoost模型的评估指标，设置为"mlogloss"表示使用多分类对数损失作为评估指标，这个指标可以衡量模型预测概率与实际类别之间的差距，值越小表示模型性能越好。
        seed=42,
    )
    
    # LightGBM参数
    LGB_PARAMS = dict(
        n_estimators=3000, # 这个参数是LightGBM模型的迭代次数，设置为3000表示模型会进行3000次迭代来训练，这个值需要根据学习率和数据规模进行调整，过多的迭代可能导致过拟合，而过少的迭代可能导致欠拟合。
        learning_rate=0.02,# 这个参数是LightGBM模型的学习率，设置为0.02可以让模型在每次迭代中更稳健地更新权重，避免过快地收敛到局部最优解，同时也需要更多的迭代次数来达到最佳性能。
        num_leaves=31,# 这个参数是LightGBM模型的最大叶子节点数，设置为31可以让模型捕捉到更复杂的特征交互关系，但同时也增加了过拟合的风险，需要通过交叉验证来确定最优值。
        max_depth=6, # 这个参数是LightGBM模型的最大树深度，设置为6可以让模型捕捉到更复杂的特征交互关系，但同时也增加了过拟合的风险，需要通过交叉验证来确定最优值。
        min_child_samples=20, # 这个参数是LightGBM模型的最小子样本数，设置为20可以让模型在分裂节点时更谨慎，只有当子样本数超过20时才会进行分裂，这可以帮助模型避免过拟合，但也可能导致欠拟合，需要通过交叉验证来确定最优值。
        subsample=0.8, # 这个参数是LightGBM模型的子样本采样率，设置为0.8可以让模型在每次迭代中随机采样80%的训练数据，增加模型的泛化能力，减少过拟合的风险。
        colsample_bytree=0.7, # 这个参数是LightGBM模型的列采样率，设置为0.7可以让模型在每次迭代中随机采样70%的特征，增加模型的泛化能力，减少过拟合的风险，同时也可以提高训练速度。
        reg_alpha=0.1, # 这个参数是LightGBM模型的L1正则化项系数，设置为0.1可以让模型在训练过程中对特征权重进行稀疏化处理，帮助模型选择重要特征，减少过拟合的风险。
        reg_lambda=0.1, # 这个参数是LightGBM模型的L2正则化项系数，设置为0.1可以让模型在训练过程中对特征权重进行平滑处理，帮助模型减少过拟合的风险，同时也可以提高模型的稳定性。
        objective="multiclass", # 这个参数是LightGBM模型的目标函数，设置为"multiclass"表示这是一个多分类问题，模型会输出每个类别的概率分布，而不是直接输出类别标签，这对于评估模型性能和进行后续处理非常有用。
        num_class=3, # 这个参数是LightGBM模型的类别数量，设置为3表示目标变量有三个类别（Low、Medium、High
        random_state=42, # 这个参数是LightGBM模型的随机种子，设置为42可以确保结果的可复现性，同时也具有一定的象征意义（被称为“生命、宇宙以及一切的答案”），在机器学习中使用固定的随机种子可以帮助我们在调试和比较模型时获得一致的结果。
        verbose=-1, # 这个参数是LightGBM模型的日志输出级别，设置为-1表示关闭所有日志输出，这可以让训练过程更清爽，尤其是在进行大量迭代时，同时也可以提高训练速度。
        force_col_wise=True # 这个参数是LightGBM模型的列存储选项，设置为True表示强制使用列存储格式来训练模型，这可以提高训练速度和内存效率，尤其是在处理高维数据时，同时也可以减少过拟合的风险。
    )
    
    # CatBoost参数
    CAT_PARAMS = dict(
        iterations=2000, # 这个参数是CatBoost模型的迭代次数，设置为2000表示模型会进行2000次迭代来训练，这个值需要根据学习率和数据规模进行调整，过多的迭代可能导致过拟合，而过少的迭代可能导致欠拟合。
        learning_rate=0.02, # 这个参数是CatBoost模型的学习率，设置为0.02可以让模型在每次迭代中更稳健地更新权重，避免过快地收敛到局部最优解，同时也需要更多的迭代次数来达到最佳性能。
        depth=6, # 这个参数是CatBoost模型的树深度，设置为6可以让模型捕捉到更复杂的特征交互关系，但同时也增加了过拟合的风险，需要通过交叉验证来确定最优值。
        l2_leaf_reg=3, # 这个参数是CatBoost模型的L2正则化项系数，设置为3可以让模型在训练过程中对特征权重进行平滑处理，帮助模型减少过拟合的风险，同时也可以提高模型的稳定性。
        border_count=64, # 这个参数是CatBoost模型的边界数量，设置为64表示模型在处理数值特征时会将其分成64个区间，这可以帮助模型更好地捕捉数值特征的分布和关系，同时也可以提高模型的性能。
        loss_function="MultiClass", # 这个参数是CatBoost模型的损失函数，设置为"MultiClass"表示这是一个多分类问题，模型会输出每个类别的概率分布，而不是直接输出类别标签，这对于评估模型性能和进行后续处理非常有用。
        eval_metric="MultiClass", # 这个参数是CatBoost模型的评估指标，设置为"MultiClass"表示使用多分类指标来评估模型性能，这个指标可以衡量模型预测概率与实际类别之间的差距，值越小表示模型性能越好。
        random_seed=42, # 这个参数是CatBoost模型的随机种子，设置为42可以确保结果的可复现性，同时也具有一定的象征意义（被称为“生命、宇宙以及一切的答案”），在机器学习中使用固定的随机种子可以帮助我们在调试和比较模型时获得一致的结果。
        verbose=False, # 这个参数是CatBoost模型的日志输出选项，设置为False表示关闭所有日志输出，这可以让训练过程更清爽，尤其是在进行大量迭代时，同时也可以提高训练速度。
        task_type='CPU' # 这个参数是CatBoost模型的计算设备选项，设置为'CPU'表示使用CPU来训练模型，这对于大多数环境来说是兼容的选择，同时也可以避免在没有GPU支持的环境中出现错误，如果有GPU可用，可以将其设置为'GPU'以加快训练速度。
    )

cfg = Config()
print("✅ Config ready")

# ============================================
# 数据加载
# ============================================
train_raw = pd.read_csv(cfg.TRAIN_PATH).dropna(subset=[cfg.TARGET]) # 加载训练数据，并删除目标变量中缺失值的行，确保模型训练时使用的都是完整的数据样本。
test      = pd.read_csv(cfg.TEST_PATH)
original  = pd.read_csv(cfg.ORIGINAL_PATH).rename(columns={"Irrigation_Requirement": cfg.TARGET}) # 加载原始数据集，并将目标变量列重命名为cfg.TARGET，以便后续处理和模型训练时使用统一的列名。

original["id"] = range(train_raw["id"].max() + 1, train_raw["id"].max() + 1 + len(original)) # 
train     = pd.concat([train_raw, original], ignore_index=True)
n_comp    = len(train_raw)

print(f"Competition train : {len(train_raw):>7,}")
print(f"Original dataset  : {len(original):>7,}")
print(f"Combined train    : {len(train):>7,}")
print(f"Test              : {len(test):>7,}")

# ============================================
# 🔥 增强特征工程
# ============================================
def add_binary_flags(df):
    df["soil_lt_25"]   = (df["Soil_Moisture"]      < cfg.SOIL_THRESH).astype(int)
    df["rain_lt_300"]  = (df["Rainfall_mm"]         < cfg.RAIN_THRESH).astype(int)
    df["temp_gt_30"]   = (df["Temperature_C"]       > cfg.TEMP_THRESH).astype(int)
    df["wind_gt_10"]   = (df["Wind_Speed_kmh"]      > cfg.WIND_THRESH).astype(int)
    df["is_harvest"]   = (df["Crop_Growth_Stage"]  == "Harvest").astype(int)
    df["is_sowing"]    = (df["Crop_Growth_Stage"]  == "Sowing").astype(int)
    df["mulching_yes"] = (df["Mulching_Used"]       == "Yes").astype(int)
    return df

def add_magic_score(df):
    high = 2 * df["soil_lt_25"] + 2 * df["rain_lt_300"] + df["temp_gt_30"] + df["wind_gt_10"]
    low  = 2 * df["is_harvest"] + 2 * df["is_sowing"] + df["mulching_yes"]
    df["magic_score"] = high - low
    df["dist_boundary_0"] = (df["magic_score"] - 0).abs()
    df["dist_boundary_3"] = (df["magic_score"] - 3).abs()
    return df

def add_decimal_digits(df):
    cols = ["Soil_Moisture", "Temperature_C", "Rainfall_mm", "Wind_Speed_kmh",
            "Humidity", "Soil_pH", "Organic_Carbon", "Electrical_Conductivity",
            "Sunlight_Hours", "Field_Area_hectare", "Previous_Irrigation_mm"]
    for col in cols:
        v = df[col].values
        df[f"{col}_dec"] = np.floor((v - np.floor(v)) * 10).astype(int)
    return df

def add_threshold_distances(df):
    df["soil_dist_25"]  = df["Soil_Moisture"]  - cfg.SOIL_THRESH
    df["rain_dist_300"] = df["Rainfall_mm"]   - cfg.RAIN_THRESH
    df["temp_dist_30"]  = df["Temperature_C"] - cfg.TEMP_THRESH
    df["wind_dist_10"]  = df["Wind_Speed_kmh"] - cfg.WIND_THRESH
    return df

def add_logit_scores(df):
    flags = {
        "Flowering":  (df["Crop_Growth_Stage"] == "Flowering").astype(float),
        "Harvest":    (df["Crop_Growth_Stage"] == "Harvest").astype(float),
        "Sowing":     (df["Crop_Growth_Stage"] == "Sowing").astype(float),
        "Vegetative": (df["Crop_Growth_Stage"] == "Vegetative").astype(float),
        "Mulch_No":   (df["Mulching_Used"]     == "No").astype(float),
        "Mulch_Yes":  (df["Mulching_Used"]     == "Yes").astype(float),
    }
    binary = {
        "soil_lt_25": df["soil_lt_25"].astype(float),
        "temp_gt_30": df["temp_gt_30"].astype(float),
        "rain_lt_300": df["rain_lt_300"].astype(float),
        "wind_gt_10": df["wind_gt_10"].astype(float),
    }
    for cls, coefs in cfg.LOGIT_COEFS.items():
        df[f"logit_{cls}"] = coefs["intercept"]
        for k, v in binary.items(): df[f"logit_{cls}"] += coefs[k] * v
        for k, v in flags.items(): df[f"logit_{cls}"] += coefs[k] * v
    return df

def add_domain_features(df):
    rain, prev, temp, sun = df["Rainfall_mm"], df["Previous_Irrigation_mm"], df["Temperature_C"], df["Sunlight_Hours"]
    humid, wind, moist    = df["Humidity"], df["Wind_Speed_kmh"], df["Soil_Moisture"]
    oc, ec, area, ph      = df["Organic_Carbon"], df["Electrical_Conductivity"], df["Field_Area_hectare"], df["Soil_pH"]
    mulch                 = df["Mulching_Used"].map({"Yes":1, "No":0}).fillna(0)

    et_proxy    = (temp * sun) / (humid + 1)
    total_water = rain + prev

    df["Total_Water_Input"] = total_water
    df["Moisture_Deficit"]  = 100 - moist
    df["Irrigation_Ratio"]  = prev / (rain + 1)
    df["ET_Proxy"]          = et_proxy
    df["Evap_Stress"]       = (temp * wind) / (humid + 1)
    df["Net_Water_Need"]    = et_proxy - rain/10
    df["VPD_Proxy"]         = temp * (1 - humid/100)
    df["Heat_Stress"]       = temp * (100-humid)/100
    df["Dryness_Index"]     = temp * sun / (rain + 1)
    df["Drought_Risk"]      = df["Dryness_Index"] * df["Moisture_Deficit"] / 100
    
    # 🔥 新增高阶特征
    df["Water_Stress"] = df["ET_Proxy"] - df["Total_Water_Input"]
    df["Soil_Quality"] = (oc * 10) / (ph * ec + 1)
    df["Climate_Index"] = (temp * humid) / (wind * rain + 1)
    df["Irrigation_Efficiency"] = prev / (df["ET_Proxy"] + 1)
    df["Water_Deficit_Ratio"] = df["Moisture_Deficit"] / (df["Total_Water_Input"] + 1)
    
    return df

def add_nonlinear_transforms(df):
    """非线性变换特征"""
    for col in cfg.NUM_COLS:
        # 对数变换
        min_val = df[col].min()
        if min_val < 0:
            df[f"{col}_log"] = np.log1p(df[col] - min_val + 1)
        else:
            df[f"{col}_log"] = np.log1p(df[col])
        
        # 平方和平方根
        df[f"{col}_square"] = df[col] ** 2
        df[f"{col}_sqrt"] = np.sqrt(np.abs(df[col]))
        
        # 分箱特征
        df[f"{col}_bin"] = pd.cut(df[col], bins=10, labels=False)
    
    return df

def add_interaction_features(df):
    """高阶交互特征"""
    # 三阶交互
    key_numerical = ['Soil_Moisture', 'Rainfall_mm', 'Temperature_C', 'Humidity', 'Wind_Speed_kmh']
    for cols in combinations(key_numerical, 3):
        name = f"{cols[0]}_{cols[1]}_{cols[2]}_interact"
        df[name] = (df[cols[0]] * df[cols[1]] * df[cols[2]]) / 1000
    
    # 比率特征
    df["Temp_Rain_Ratio"] = df["Temperature_C"] / (df["Rainfall_mm"] + 1)
    df["Moisture_Temp_Ratio"] = df["Soil_Moisture"] / (df["Temperature_C"] + 1)
    df["Wind_Humid_Ratio"] = df["Wind_Speed_kmh"] / (df["Humidity"] + 1)
    
    return df

def add_pca_features(train_df, test_df, n_components=5):
    """PCA降维特征（先填充缺失值）"""
    # 选择数值列
    num_cols = train_df[cfg.NUM_COLS].select_dtypes(include=[np.number]).columns.tolist()
    
    # 合并训练集和测试集
    combined = pd.concat([train_df[num_cols], test_df[num_cols]], axis=0)
    
    # 🔥 先填充缺失值（使用中位数）
    from sklearn.impute import SimpleImputer
    imputer = SimpleImputer(strategy='median')
    combined_imputed = imputer.fit_transform(combined)
    
    # 标准化
    scaler = StandardScaler()
    combined_scaled = scaler.fit_transform(combined_imputed)
    
    # PCA
    pca = PCA(n_components=n_components, random_state=cfg.RANDOM_SEED)
    combined_pca = pca.fit_transform(combined_scaled)
    
    n_train = len(train_df)
    for i in range(n_components):
        train_df[f"pca_{i}"] = combined_pca[:n_train, i]
        test_df[f"pca_{i}"] = combined_pca[n_train:, i]
    
    print(f"PCA explained variance ratio: {pca.explained_variance_ratio_.sum():.3f}")
    return train_df, test_df

def engineer_features(df, is_train=True):
    df = df.copy()
    df = add_binary_flags(df)
    df = add_magic_score(df)
    df = add_decimal_digits(df)
    df = add_threshold_distances(df)
    df = add_logit_scores(df)
    df = add_domain_features(df)
    df = add_nonlinear_transforms(df)
    df = add_interaction_features(df)
    return df

# 特征工程
print("🔧 开始特征工程...")
train_eng = engineer_features(train)
test_eng  = engineer_features(test)

# PCA特征
train_eng, test_eng = add_pca_features(train_eng, test_eng, n_components=5)

print(f"Train shape: {train_eng.shape}, Test shape: {test_eng.shape}")

# ============================================
# 类别编码
# ============================================
for col in cfg.CAT_COLS:
    le = LabelEncoder()
    le.fit(pd.concat([train_eng[col].astype(str), test_eng[col].astype(str)]))
    train_eng[col] = le.transform(train_eng[col].astype(str)).astype("int32")
    test_eng[col]  = le.transform(test_eng[col].astype(str)).astype("int32")

# ============================================
# 准备特征矩阵
# ============================================
y = train_eng[cfg.TARGET].map(cfg.TARGET_MAPPING).values

# 排除非特征列
exclude_cols = ["id", cfg.TARGET]
feature_cols = [c for c in train_eng.columns if c not in exclude_cols and train_eng[c].dtype != object]

X = train_eng[feature_cols].fillna(train_eng[feature_cols].median()).astype("float32")
X_test = test_eng[feature_cols].fillna(test_eng[feature_cols].median()).astype("float32")

print(f"Final feature matrix: {X.shape}")

# ============================================
# 🔥 多模型集成训练
# ============================================
class EnsembleModel:
    def __init__(self):
        self.xgb_models = []
        self.lgb_models = []
        self.cat_models = []
        self.meta_model = None
        self.feature_selector = None
        
    def fit(self, X, y, X_test):
        skf = StratifiedKFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.RANDOM_SEED)
        
        # OOF预测存储
        oof_xgb = np.zeros((len(X), 3))
        oof_lgb = np.zeros((len(X), 3))
        oof_cat = np.zeros((len(X), 3))
        
        test_xgb = np.zeros((len(X_test), 3))
        test_lgb = np.zeros((len(X_test), 3))
        test_cat = np.zeros((len(X_test), 3))
        
        # 特征选择
        print("\n🔍 特征选择中...")
        selector = SelectFromModel(
            xgb.XGBClassifier(**cfg.XGB_PARAMS, n_estimators=100),
            threshold='median'
        )
        X_selected = selector.fit_transform(X, y)
        X_test_selected = selector.transform(X_test)
        self.feature_selector = selector
        print(f"选择特征数: {X_selected.shape[1]}")
        
        # SMOTE过采样
        smote = SMOTE(random_state=cfg.RANDOM_SEED)
        
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_selected, y)):
            print(f"\n📁 Fold {fold+1}/{cfg.N_FOLDS}")
            
            X_tr, X_val = X_selected[tr_idx], X_selected[val_idx]
            y_tr, y_val = y[tr_idx], y[val_idx]
            
            # SMOTE
            X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)
            
            # XGBoost
            print("  Training XGBoost...")
            xgb_model = xgb.XGBClassifier(**cfg.XGB_PARAMS, n_estimators=2000)
            xgb_model.fit(
                X_tr_res, y_tr_res,
                eval_set=[(X_val, y_val)],
                verbose=False
            )
            self.xgb_models.append(xgb_model)
            oof_xgb[val_idx] = xgb_model.predict_proba(X_val)
            test_xgb += xgb_model.predict_proba(X_test_selected) / cfg.N_FOLDS
            
            # LightGBM
            print("  Training LightGBM...")
            lgb_model = lgb.LGBMClassifier(**cfg.LGB_PARAMS)
            lgb_model.fit(
                X_tr_res, y_tr_res,
                eval_set=[(X_val, y_val)],
                callbacks=[lgb.early_stopping(100, verbose=False)]
            )
            self.lgb_models.append(lgb_model)
            oof_lgb[val_idx] = lgb_model.predict_proba(X_val)
            test_lgb += lgb_model.predict_proba(X_test_selected) / cfg.N_FOLDS
            
            # CatBoost
            print("  Training CatBoost...")
            cat_model = CatBoostClassifier(**cfg.CAT_PARAMS)
            cat_model.fit(
                X_tr_res, y_tr_res,
                eval_set=[(X_val, y_val)],
                early_stopping_rounds=100,
                verbose=False
            )
            self.cat_models.append(cat_model)
            oof_cat[val_idx] = cat_model.predict_proba(X_val)
            test_cat += cat_model.predict_proba(X_test_selected) / cfg.N_FOLDS
            
            gc.collect()
        
        # 训练元模型（Stacking）
        print("\n🎯 训练Stacking元模型...")
        meta_features = np.hstack([oof_xgb, oof_lgb, oof_cat])
        self.meta_model = LogisticRegression(C=0.1, multi_class='multinomial', max_iter=1000)
        self.meta_model.fit(meta_features, y)
        
        # 计算OOF分数
        meta_oof = self.meta_model.predict_proba(meta_features)
        ba_score = balanced_accuracy_score(y, meta_oof.argmax(axis=1))
        print(f"\n📊 OOF Balanced Accuracy: {ba_score:.5f}")
        
        # 测试集预测
        test_meta = np.hstack([test_xgb, test_lgb, test_cat])
        test_probs = self.meta_model.predict_proba(test_meta)
        
        return test_probs, meta_oof
    
    def predict(self, X_test):
        X_test_selected = self.feature_selector.transform(X_test)
        
        test_xgb = np.zeros((len(X_test), 3))
        test_lgb = np.zeros((len(X_test), 3))
        test_cat = np.zeros((len(X_test), 3))
        
        for model in self.xgb_models:
            test_xgb += model.predict_proba(X_test_selected) / len(self.xgb_models)
        for model in self.lgb_models:
            test_lgb += model.predict_proba(X_test_selected) / len(self.lgb_models)
        for model in self.cat_models:
            test_cat += model.predict_proba(X_test_selected) / len(self.cat_models)
        
        test_meta = np.hstack([test_xgb, test_lgb, test_cat])
        return self.meta_model.predict_proba(test_meta)

# 训练集成模型
print("\n🚀 开始训练集成模型...")
ensemble = EnsembleModel()
test_probs, oof_probs = ensemble.fit(X, y, X_test)

# ============================================
# 🔥 迭代伪标签优化
# ============================================
print("\n🏷️ 开始迭代伪标签...")
best_probs = test_probs.copy()
best_score = balanced_accuracy_score(y, oof_probs.argmax(axis=1))

for iteration in range(cfg.N_PSEUDO_ITER):
    # 选择高置信度样本
    conf_threshold = np.percentile(test_probs.max(axis=1), 95 - iteration*2)
    pseudo_mask = test_probs.max(axis=1) >= conf_threshold
    n_pseudo = pseudo_mask.sum()
    
    if n_pseudo < 100:
        break
    
    print(f"Iteration {iteration+1}: {n_pseudo} pseudo-labels (threshold={conf_threshold:.3f})")
    
    # 扩充训练集
    X_aug = np.vstack([X, X_test[pseudo_mask]])
    y_aug = np.concatenate([y, test_probs[pseudo_mask].argmax(axis=1)])
    
    # 重新训练
    ensemble = EnsembleModel()
    test_probs_new, _ = ensemble.fit(X_aug, y_aug, X_test)
    
    # 评估
    score_new = balanced_accuracy_score(y, oof_probs.argmax(axis=1))
    if score_new > best_score:
        best_score = score_new
        best_probs = test_probs_new
        print(f"  ✅ 分数提升至 {best_score:.5f}")
    else:
        print(f"  ⏸️ 分数未提升，保持 {best_score:.5f}")

# ============================================
# 🔥 类别权重优化
# ============================================
def optimize_weights(probs, y_true):
    def neg_bal_acc(w):
        preds = (probs * w).argmax(axis=1)
        return -balanced_accuracy_score(y_true, preds)
    
    initial_weights = [0.9, 1.0, 1.1]
    result = minimize(
        neg_bal_acc, 
        initial_weights, 
        method='Nelder-Mead',
        options={'maxiter': 1000}
    )
    return result.x

print("\n⚖️ 优化类别权重...")
optimal_weights = optimize_weights(oof_probs, y)
print(f"最优权重: {optimal_weights}")

# 应用权重
final_probs = best_probs * optimal_weights
final_preds = final_probs.argmax(axis=1)

# ============================================
# 生成提交文件
# ============================================
submission = pd.DataFrame({
    'id': test['id'],
    cfg.TARGET: [cfg.INV_MAPPING[p] for p in final_preds]
})

print("\n📈 预测分布:")
print(submission[cfg.TARGET].value_counts())

submission.to_csv('submission_optimized.csv', index=False)
print("\n✅ 优化版提交文件已生成: submission_optimized.csv")

# ============================================
# 保存模型（可选）
# ============================================
with open('ensemble_model.pkl', 'wb') as f:
    pickle.dump(ensemble, f)
print("✅ 模型已保存: ensemble_model.pkl")

# ============================================
# 特征重要性分析
# ============================================
print("\n📊 特征重要性分析（XGBoost）:")
importance = ensemble.xgb_models[0].feature_importances_
selected_features = X.columns[ensemble.feature_selector.get_support()]
top_features = pd.DataFrame({
    'feature': selected_features,
    'importance': importance[:len(selected_features)]
}).sort_values('importance', ascending=False).head(15)

print(top_features.to_string(index=False))

✅ Imports complete
✅ Config ready
Competition train : 630,000
Original dataset  : 270,000
Combined train    : 900,000
Test              : 270,000
🔧 开始特征工程...


KeyboardInterrupt: 

## v9

In [4]:
# Imports & Configuration
import gc
import warnings
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import xgboost as xgb
from scipy.optimize import minimize
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, TargetEncoder
from sklearn.utils.class_weight import compute_sample_weight
from tqdm import tqdm

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
print("✅ Imports complete")

class Config:
    TRAIN_PATH    = "train.csv"
    TEST_PATH     = "test.csv"
    ORIGINAL_PATH = "sample_submission.csv"

    TARGET         = "Irrigation_Need"
    TARGET_MAPPING = {"Low": 0, "Medium": 1, "High": 2}
    INV_MAPPING    = {0: "Low", 1: "Medium", 2: "High"}

    N_FOLDS       = 5
    RANDOM_SEED   = 42 # 固定随机种子以确保结果可复现
    PSEUDO_THRESH = 0.95 # 伪标签置信度阈值

    SOIL_THRESH = 25 # 土壤湿度阈值，低于25可能表示干旱条件，增加灌溉需求
    RAIN_THRESH = 300
    TEMP_THRESH = 30
    WIND_THRESH = 10

    CAT_COLS = [
        "Soil_Type", "Crop_Type", "Crop_Growth_Stage", "Season",
        "Irrigation_Type", "Water_Source", "Mulching_Used", "Region",
    ]
    NUM_COLS = [
        "Soil_pH", "Soil_Moisture", "Organic_Carbon", "Electrical_Conductivity",
        "Temperature_C", "Humidity", "Rainfall_mm", "Sunlight_Hours",
        "Wind_Speed_kmh", "Field_Area_hectare", "Previous_Irrigation_mm",
    ]

    LOGIT_COEFS = {
        "Low": {
            "intercept": 16.3173, "soil_lt_25": -11.0237, "temp_gt_30": -5.8559,
            "rain_lt_300": -10.8500, "wind_gt_10": -5.8284,
            "Flowering": -5.4155, "Harvest": 5.5073, "Sowing": 5.2299, "Vegetative": -5.4617,
            "Mulch_No": -3.0014, "Mulch_Yes": 2.8613,
        },
        "Medium": {
            "intercept": 4.6524, "soil_lt_25": 0.3290, "temp_gt_30": -0.0204,
            "rain_lt_300": 0.1542, "wind_gt_10": 0.0841,
            "Flowering": 0.3586, "Harvest": -0.1348, "Sowing": -0.3547, "Vegetative": 0.3334,
            "Mulch_No": 0.1883, "Mulch_Yes": 0.0142,
        },
        "High": {
            "intercept": -20.9697, "soil_lt_25": 10.6947, "temp_gt_30": 5.8763,
            "rain_lt_300": 10.6958, "wind_gt_10": 5.7444,
            "Flowering": 5.0569, "Harvest": -5.3725, "Sowing": -4.8752, "Vegetative": 5.1283,
            "Mulch_No": 2.8131, "Mulch_Yes": -2.8755,
        },
    }

    XGB_PARAMS = dict(
        max_depth=5,
        learning_rate=0.02,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.6,
        gamma=2.0,
        reg_alpha=0.01,
        reg_lambda=1.0,
        objective="multi:softprob",
        num_class=3,
        tree_method="hist",
        enable_categorical=True,
        eval_metric="mlogloss",
        seed=42,
    )

cfg = Config()
print("✅ Config ready")

# Data Loading
train_raw = pd.read_csv(cfg.TRAIN_PATH).dropna(subset=[cfg.TARGET])
test      = pd.read_csv(cfg.TEST_PATH)
original  = pd.read_csv(cfg.ORIGINAL_PATH).rename(columns={"Irrigation_Requirement": cfg.TARGET})

original["id"] = range(train_raw["id"].max() + 1, train_raw["id"].max() + 1 + len(original))
train     = pd.concat([train_raw, original], ignore_index=True)
n_comp    = len(train_raw)

print(f"Competition train : {len(train_raw):>7,}")
print(f"Original dataset  : {len(original):>7,}")
print(f"Combined train    : {len(train):>7,}")
print(f"Test              : {len(test):>7,}")

# ==========================
# 🔥 高分核心：超强特征工程
# ==========================
def add_binary_flags(df):
    df["soil_lt_25"]   = (df["Soil_Moisture"]      < cfg.SOIL_THRESH).astype(int)
    df["rain_lt_300"]  = (df["Rainfall_mm"]         < cfg.RAIN_THRESH).astype(int)
    df["temp_gt_30"]   = (df["Temperature_C"]       > cfg.TEMP_THRESH).astype(int)
    df["wind_gt_10"]   = (df["Wind_Speed_kmh"]      > cfg.WIND_THRESH).astype(int)
    df["is_harvest"]   = (df["Crop_Growth_Stage"]  == "Harvest").astype(int)
    df["is_sowing"]    = (df["Crop_Growth_Stage"]  == "Sowing").astype(int)
    df["mulching_yes"] = (df["Mulching_Used"]       == "Yes").astype(int)
    return df

def add_magic_score(df):
    high = 2 * df["soil_lt_25"] + 2 * df["rain_lt_300"] + df["temp_gt_30"] + df["wind_gt_10"]
    low  = 2 * df["is_harvest"] + 2 * df["is_sowing"] + df["mulching_yes"]
    df["magic_score"] = high - low
    df["dist_boundary_0"] = (df["magic_score"] - 0).abs()
    df["dist_boundary_3"] = (df["magic_score"] - 3).abs()
    return df

def add_decimal_digits(df):
    cols = ["Soil_Moisture", "Temperature_C", "Rainfall_mm", "Wind_Speed_kmh",
            "Humidity", "Soil_pH", "Organic_Carbon", "Electrical_Conductivity",
            "Sunlight_Hours", "Field_Area_hectare", "Previous_Irrigation_mm"]
    for col in cols:
        v = df[col].values
        df[f"{col}_dec"] = np.floor((v - np.floor(v)) * 10).astype(int)
    return df

def add_threshold_distances(df):
    df["soil_dist_25"]  = df["Soil_Moisture"]  - cfg.SOIL_THRESH
    df["rain_dist_300"] = df["Rainfall_mm"]   - cfg.RAIN_THRESH
    df["temp_dist_30"]  = df["Temperature_C"] - cfg.TEMP_THRESH
    df["wind_dist_10"]  = df["Wind_Speed_kmh"] - cfg.WIND_THRESH
    return df

def add_logit_scores(df):
    flags = {
        "Flowering":  (df["Crop_Growth_Stage"] == "Flowering").astype(float),
        "Harvest":    (df["Crop_Growth_Stage"] == "Harvest").astype(float),
        "Sowing":     (df["Crop_Growth_Stage"] == "Sowing").astype(float),
        "Vegetative": (df["Crop_Growth_Stage"] == "Vegetative").astype(float),
        "Mulch_No":   (df["Mulching_Used"]     == "No").astype(float),
        "Mulch_Yes":  (df["Mulching_Used"]     == "Yes").astype(float),
    }
    binary = {
        "soil_lt_25": df["soil_lt_25"].astype(float),
        "temp_gt_30": df["temp_gt_30"].astype(float),
        "rain_lt_300": df["rain_lt_300"].astype(float),
        "wind_gt_10": df["wind_gt_10"].astype(float),
    }
    for cls, coefs in cfg.LOGIT_COEFS.items():
        df[f"logit_{cls}"] = coefs["intercept"]
        for k, v in binary.items(): df[f"logit_{cls}"] += coefs[k] * v
        for k, v in flags.items(): df[f"logit_{cls}"] += coefs[k] * v
    return df

def add_domain_features(df):
    rain, prev, temp, sun = df["Rainfall_mm"], df["Previous_Irrigation_mm"], df["Temperature_C"], df["Sunlight_Hours"]
    humid, wind, moist    = df["Humidity"], df["Wind_Speed_kmh"], df["Soil_Moisture"]
    oc, ec, area, ph      = df["Organic_Carbon"], df["Electrical_Conductivity"], df["Field_Area_hectare"], df["Soil_pH"]
    mulch                 = df["Mulching_Used"].map({"Yes":1, "No":0}).fillna(0)

    et_proxy    = (temp * sun) / (humid + 1)
    total_water = rain + prev

    df["Total_Water_Input"] = total_water
    df["Moisture_Deficit"]  = 100 - moist
    df["Irrigation_Ratio"]  = prev / (rain + 1)
    df["ET_Proxy"]          = et_proxy
    df["Evap_Stress"]       = (temp * wind) / (humid + 1)
    df["Net_Water_Need"]    = et_proxy - rain/10
    df["VPD_Proxy"]         = temp * (1 - humid/100)
    df["Heat_Stress"]       = temp * (100-humid)/100
    df["Dryness_Index"]     = temp * sun / (rain + 1)
    df["Drought_Risk"]      = df["Dryness_Index"] * df["Moisture_Deficit"] / 100
    return df

def engineer_features(df):
    df = df.copy()
    df = add_binary_flags(df)
    df = add_magic_score(df)
    df = add_decimal_digits(df)
    df = add_threshold_distances(df)
    df = add_logit_scores(df)
    df = add_domain_features(df)
    return df

# 特征工程
train_eng = engineer_features(train)
test_eng  = engineer_features(test)
print(f"Train shape: {train_eng.shape}, Test shape: {test_eng.shape}")

# 类别编码（无泄露版）
for col in cfg.CAT_COLS:
    le = LabelEncoder()
    le.fit(pd.concat([train_eng[col].astype(str), test_eng[col].astype(str)]))
    train_eng[col] = le.transform(train_eng[col].astype(str)).astype("int32")
    test_eng[col]  = le.transform(test_eng[col].astype(str)).astype("int32")

# 交互特征
n_train = len(train_eng)
interaction_cols = []
for c1, c2 in tqdm(combinations(cfg.NUM_COLS + cfg.CAT_COLS, 2), desc="Interactions"):
    name = f"{c1}|{c2}"
    combined = pd.concat([
        train_eng[c1].astype(str)+"_"+train_eng[c2].astype(str),
        test_eng[c1].astype(str)+"_"+test_eng[c2].astype(str)
    ])
    codes, _ = combined.factorize()
    if pd.Series(codes).nunique() > len(codes)//2: continue
    train_eng[name] = codes[:n_train]
    test_eng[name]  = codes[n_train:]
    interaction_cols.append(name)

print(f"Created {len(interaction_cols)} interaction features")

# 构建特征矩阵
y = train_eng[cfg.TARGET].map(cfg.TARGET_MAPPING).values
drop_cols = {"id", cfg.TARGET, *interaction_cols}
base_feats = [c for c in train_eng.columns if c not in drop_cols and train_eng[c].dtype!=object]
med = train_eng[base_feats].median()

X_base  = train_eng[base_feats].fillna(med).astype("float32")
X_tbase = test_eng[base_feats].fillna(med).astype("float32")
X_pair  = train_eng[interaction_cols]
X_tpair = test_eng[interaction_cols]

# ==========================
# 🔥 10折CV + 目标编码
# ==========================
def apply_target_encoding(X_tr_p, y_tr, X_va_p, X_te_p, cols):
    enc = TargetEncoder(target_type="multiclass", cv=5, random_state=cfg.RANDOM_SEED)
    tr_enc = pd.DataFrame(enc.fit_transform(X_tr_p[cols], y_tr))
    va_enc = pd.DataFrame(enc.transform(X_va_p[cols]))
    te_enc = pd.DataFrame(enc.transform(X_te_p[cols]))
    return tr_enc, va_enc, te_enc

def balanced_accuracy_metric(preds, dmatrix):
    labels = dmatrix.get_label().astype(int)
    y_pred = preds.reshape(-1,3).argmax(axis=1)
    return "bal_ACC", balanced_accuracy_score(labels, y_pred)

skf = StratifiedKFold(n_splits=cfg.N_FOLDS, shuffle=True, random_state=cfg.RANDOM_SEED)
oof_probs  = np.zeros((len(X_base), 3))
test_probs = np.zeros((len(X_tbase), 3))
best_iters = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_base, y)):
    print(f"\nFold {fold+1}/{cfg.N_FOLDS}")
    tr_enc, va_enc, te_enc = apply_target_encoding(
        X_pair.iloc[tr_idx], y[tr_idx], X_pair.iloc[val_idx], X_tpair, interaction_cols
    )
    Xt = pd.concat([X_base.iloc[tr_idx].reset_index(drop=True), tr_enc], axis=1)
    Xv = pd.concat([X_base.iloc[val_idx].reset_index(drop=True), va_enc], axis=1)
    Xe = pd.concat([X_tbase.reset_index(drop=True), te_enc], axis=1)

    dtrain = xgb.DMatrix(Xt, label=y[tr_idx])
    dval   = xgb.DMatrix(Xv, label=y[val_idx])
    dtest  = xgb.DMatrix(Xe)

    model = xgb.train(
        cfg.XGB_PARAMS, dtrain, 10000,
        evals=[(dtrain,"train"),(dval,"val")],
        custom_metric=balanced_accuracy_metric, maximize=True,
        early_stopping_rounds=300, verbose_eval=500
    )
    oof_probs[val_idx] = model.predict(dval)
    test_probs += model.predict(dtest) / cfg.N_FOLDS
    best_iters.append(model.best_iteration)
    gc.collect()

baseline_ba = balanced_accuracy_score(y, oof_probs.argmax(axis=1))
print(f"\nOOF Balanced Accuracy: {baseline_ba:.5f}")

# ==========================
# 🔥 伪标签（高分关键）
# ==========================
mask = test_probs.max(axis=1) >= cfg.PSEUDO_THRESH
print(f"Pseudo-label samples: {mask.sum()}")

if mask.sum() >= 500:
    tr_enc_full, _, te_enc_full = apply_target_encoding(X_pair, y, X_pair, X_tpair, interaction_cols)
    X_full   = pd.concat([X_base, tr_enc_full], axis=1)
    X_test_f = pd.concat([X_tbase, te_enc_full], axis=1)
    X_aug = pd.concat([X_full, X_test_f[mask]], ignore_index=True)
    y_aug = np.concatenate([y, test_probs[mask].argmax(axis=1)])

    final_model = xgb.XGBClassifier(**cfg.XGB_PARAMS, n_estimators=int(np.mean(best_iters)))
    final_model.fit(X_aug, y_aug, verbose=False)
    final_test_probs = final_model.predict_proba(X_test_f)
else:
    final_test_probs = test_probs

# ==========================
# 🔥 类别权重优化（直接涨0.01+）
# ==========================
def neg_bal_acc(w, probs, y):
    return -balanced_accuracy_score(y, (probs * w).argmax(axis=1))

best_w = [0.9, 1.0, 5.0]
res = minimize(neg_bal_acc, x0=best_w, args=(oof_probs, y), method="Nelder-Mead")
final_w = res.x if -res.fun > baseline_ba else np.array(best_w)
opt_ba = balanced_accuracy_score(y, (oof_probs * final_w).argmax(axis=1))

print(f"Optimized BA: {opt_ba:.5f}")
print(f"Weight: {np.round(final_w, 4)}")

# 提交
test_preds = (final_test_probs * final_w).argmax(axis=1)
sub = pd.DataFrame({
    "id": test["id"],
    cfg.TARGET: [cfg.INV_MAPPING[p] for p in test_preds]
})
sub.to_csv("submission.csv", index=False)
print("✅ 提交文件已生成！")
print(sub[cfg.TARGET].value_counts())

✅ Imports complete
✅ Config ready
Competition train : 630,000
Original dataset  : 270,000
Combined train    : 900,000
Test              : 270,000
Train shape: (900000, 59), Test shape: (270000, 58)


Interactions: 171it [01:17,  2.21it/s]


Created 144 interaction features

Fold 1/5
[0]	train-mlogloss:0.65445	train-bal_ACC:0.33333	val-mlogloss:0.65442	val-bal_ACC:0.33333
[500]	train-mlogloss:0.03048	train-bal_ACC:0.97450	val-mlogloss:0.03318	val-bal_ACC:0.97169
[1000]	train-mlogloss:0.02516	train-bal_ACC:0.98012	val-mlogloss:0.03207	val-bal_ACC:0.97286
[1500]	train-mlogloss:0.02153	train-bal_ACC:0.98367	val-mlogloss:0.03190	val-bal_ACC:0.97296
[1573]	train-mlogloss:0.02113	train-bal_ACC:0.98403	val-mlogloss:0.03189	val-bal_ACC:0.97297

Fold 2/5
[0]	train-mlogloss:0.65443	train-bal_ACC:0.33333	val-mlogloss:0.65446	val-bal_ACC:0.33333
[500]	train-mlogloss:0.03027	train-bal_ACC:0.97471	val-mlogloss:0.03388	val-bal_ACC:0.97224
[1000]	train-mlogloss:0.02494	train-bal_ACC:0.98014	val-mlogloss:0.03283	val-bal_ACC:0.97291
[1236]	train-mlogloss:0.02305	train-bal_ACC:0.98222	val-mlogloss:0.03273	val-bal_ACC:0.97270

Fold 3/5
[0]	train-mlogloss:0.65444	train-bal_ACC:0.33333	val-mlogloss:0.65443	val-bal_ACC:0.33333
[500]	train-mloglo

## v10=

In [1]:
# ============================================
# 🔥 XGBoost Trio + Optuna 自动调参 - 本地版
# ============================================

%pip install optuna xgboost -q

import gc
import random
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from pandas.api.types import is_object_dtype, is_string_dtype, is_categorical_dtype
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb
from xgboost import XGBClassifier
import optuna
from optuna.samplers import TPESampler

warnings.filterwarnings("ignore")

# ============================================
# 全局配置
# ============================================
SEED = 2026
SEEDS = [2026, 3407]
N_FOLDS = 5
TARGET_COL = "Irrigation_Need"
ID_COL = "id"
N_CLASSES = 3
USE_GPU = False  # 🔥 本地改为 False，除非你有 GPU
OPTUNA_TRIALS = 30  # 🔥 减少试验次数，加速

def seed_everything(seed: int = 2026) -> None:
    random.seed(seed)
    np.random.seed(seed)

seed_everything(SEED)
print("✅ Config loaded")

# ============================================
# 加载本地数据
# ============================================
DATA_DIR = Path(".")

# 🔥 改为本地文件路径
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sub = pd.read_csv("sample_submission.csv")

# 如果有额外的原始数据集
original = pd.read_csv("sample_submission.csv").rename(columns={"Irrigation_Requirement": TARGET_COL})
original["id"] = range(train["id"].max() + 1, train["id"].max() + 1 + len(original))
train = pd.concat([train, original], ignore_index=True)

print(f"train: {train.shape}")
print(f"test : {test.shape}")

# 目标编码
label2idx = {"Low": 0, "Medium": 1, "High": 2}
idx2label = {v: k for k, v in label2idx.items()}
train[TARGET_COL] = train[TARGET_COL].map(label2idx).astype("int8")

test_ids = test[ID_COL].copy()
y = train[TARGET_COL].copy()

X_train_raw = train.drop(columns=[TARGET_COL]).copy()
X_test_raw = test.copy()

if ID_COL in X_train_raw.columns:
    X_train_raw = X_train_raw.drop(columns=[ID_COL])
if ID_COL in X_test_raw.columns:
    X_test_raw = X_test_raw.drop(columns=[ID_COL])

print("y distribution:")
print(y.value_counts(normalize=True).sort_index())

# ============================================
# 特征工程
# ============================================
def get_base_cols(df):
    cats = [c for c in df.columns if is_object_dtype(df[c]) or is_string_dtype(df[c]) or is_categorical_dtype(df[c])]
    nums = [c for c in df.columns if c not in cats]
    return cats, nums

def FE(df, M):
    out = df.copy()
    for c in NUMS_BASE:
        for k in range(-4, 3):
            out[f"{c}_digit{k}"] = (out[c] // (10**k) % 10).astype('int8')
        if M[c] < 10:
            out[c] = out[c].round(3)
        elif M[c] < 100:
            out[c] = out[c].round(2)
        else:
            out[c] = out[c].round(1)
    return out

CATS_BASE, NUMS_BASE = get_base_cols(X_train_raw)
M_vals = X_train_raw[NUMS_BASE].max()
X_train_fe = FE(X_train_raw, M_vals)
X_test_fe = FE(X_test_raw, M_vals)

# 删除常量列
DROP = [c for c in X_test_fe.columns if X_test_fe[c].nunique() == 1]
print(f"Dropping constant cols: {len(DROP)}")
X_train_fe.drop(columns=DROP, inplace=True, errors='ignore')
X_test_fe.drop(columns=DROP, inplace=True, errors='ignore')

# 类别特征编码
CATEGORY = CATS_BASE + [c for c in X_test_fe.columns if 'digit' in c]
for c in CATEGORY:
    if c not in X_train_fe.columns:
        continue
    freq = X_train_fe[c].value_counts()
    mapping = {val: idx for idx, (val, count) in enumerate(freq[freq >= 5].items())}
    mapping_default = len(mapping)
    X_train_fe[c] = X_train_fe[c].map(lambda x: mapping.get(x, mapping_default))
    X_test_fe[c] = X_test_fe[c].map(lambda x: mapping.get(x, mapping_default))

FEATURES = CATEGORY + [c for c in NUMS_BASE if c in X_test_fe.columns]

print(f"Base train shape: {X_train_fe.shape}")
print(f"Base test shape : {X_test_fe.shape}")

# ============================================
# 有序目标编码 (Ordered Target Encoding)
# ============================================
class OrderedTE:
    def __init__(self, a=1):
        self.a = a
        
    def fit(self, train, category_cols=[], target_col='target'):
        self.train = train.copy()
        self.target_col = target_col
        self.category_cols = category_cols
        
        self.classes_ = sorted(train[target_col].unique())
        self.num_classes_ = len(self.classes_)
        self.global_prior_ = train[target_col].value_counts(normalize=True).sort_index().values
        
        for c in self.category_cols:
            if c not in train.columns:
                continue
            for k, cls in enumerate(self.classes_):
                y_binary = (train[target_col] == cls).astype(np.int8)
                df = train[[c]].copy()
                df['y'] = y_binary.values
                df['cnt'] = 1
                df['cum_cnt'] = df.groupby(c)['cnt'].cumsum() - df['cnt']
                df['cum_sum'] = df.groupby(c)['y'].cumsum() - df['y']
                smooth_prior = self.a * self.global_prior_[k]
                te_col = f'{c}_TE_cls{cls}'
                df[te_col] = (df['cum_sum'] + smooth_prior) / (df['cum_cnt'] + self.a)
                df.loc[df['cum_cnt'] == -1, te_col] = self.global_prior_[k]
                self.train[te_col] = df[te_col].astype(np.float32).values
                
        return self.train
    
    def transform(self, test):
        out = test.copy()
        for c in self.category_cols:
            if c not in out.columns:
                continue
            # 简化版 transform
            for k, cls in enumerate(self.classes_):
                te_col = f'{c}_TE_cls{cls}'
                if te_col in self.train.columns:
                    mean_te = self.train.groupby(c)[te_col].mean().to_dict()
                    out[te_col] = out[c].map(mean_te).fillna(self.global_prior_[k]).astype(np.float32)
        return out

# ============================================
# 样本权重
# ============================================
unique, counts = np.unique(y, return_counts=True)
weights_dict = {cls: (len(y) / len(unique)) / cnt for cls, cnt in zip(unique, counts)}
sample_weights = np.array([weights_dict[lbl] for lbl in y])

# ============================================
# Optuna 调参 - XGBoost Base
# ============================================
def objective_xgb(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 5),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.03),
        'n_estimators': trial.suggest_int('n_estimators', 2000, 3500),
        'min_child_weight': trial.suggest_int('min_child_weight', 2, 5),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 0.7),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0),
        'gamma': trial.suggest_float('gamma', 0.0, 0.1),
        'tree_method': 'hist',
        'device': 'cuda' if USE_GPU else 'cpu',
        'random_state': SEED,
        'n_jobs': -1,
        'enable_categorical': True,
    }
    
    kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = []
    
    for tr_idx, va_idx in kf.split(X_train_fe, y):
        X_tr, X_va = X_train_fe.iloc[tr_idx].copy(), X_train_fe.iloc[va_idx].copy()
        y_tr, y_va = y.iloc[tr_idx].copy(), y.iloc[va_idx].copy()
        w_tr = sample_weights[tr_idx]

        te = OrderedTE(a=1)
        tr_df = pd.concat([X_tr, y_tr], axis=1)
        tr_df['w'] = w_tr
        te_train = te.fit(tr_df.sample(frac=1, random_state=SEED), category_cols=FEATURES, target_col=TARGET_COL)
        
        X_tr_enc = te_train.drop(columns=[TARGET_COL, 'w'], errors='ignore')
        y_tr_enc = te_train[TARGET_COL].values
        w_tr_enc = te_train['w'].values
        
        X_va_enc = te.transform(X_va)
        
        # 确保列一致
        common_cols = list(set(X_tr_enc.columns) & set(X_va_enc.columns))
        X_tr_enc = X_tr_enc[common_cols]
        X_va_enc = X_va_enc[common_cols]
        
        model = XGBClassifier(**params)
        model.fit(X_tr_enc, y_tr_enc, sample_weight=w_tr_enc, verbose=False)
        va_p = model.predict_proba(X_va_enc)
        scores.append(balanced_accuracy_score(y_va, va_p.argmax(axis=1)))
        
        del model
        gc.collect()
        
    return np.mean(scores)

print("\n🔍 开始 Optuna 调参...")
study_xgb = optuna.create_study(direction='maximize', sampler=TPESampler(seed=SEED))
study_xgb.optimize(objective_xgb, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

best_xgb_params = study_xgb.best_params
best_xgb_params['tree_method'] = 'hist'
best_xgb_params['device'] = 'cuda' if USE_GPU else 'cpu'
best_xgb_params['random_state'] = SEED
best_xgb_params['n_jobs'] = -1
best_xgb_params['enable_categorical'] = True

print(f"\n✅ Best XGB Params: {best_xgb_params}")
print(f"Best CV Score: {study_xgb.best_value:.5f}")

# ============================================
# 完整 CV 训练
# ============================================
print("\n🚀 开始完整 CV 训练...")

kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_probs = np.zeros((len(y), N_CLASSES), dtype=np.float32)
test_probs = np.zeros((len(X_test_fe), N_CLASSES), dtype=np.float32)

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train_fe, y), start=1):
    print(f"\n📁 Fold {fold}/{N_FOLDS}")
    
    X_tr, X_va = X_train_fe.iloc[tr_idx].copy(), X_train_fe.iloc[va_idx].copy()
    y_tr, y_va = y.iloc[tr_idx].copy(), y.iloc[va_idx].copy()
    w_tr = sample_weights[tr_idx]

    te = OrderedTE(a=1)
    tr_df = pd.concat([X_tr, y_tr], axis=1)
    tr_df['w'] = w_tr
    te_train = te.fit(tr_df.sample(frac=1, random_state=SEED+fold), category_cols=FEATURES, target_col=TARGET_COL)
    
    X_tr_enc = te_train.drop(columns=[TARGET_COL, 'w'], errors='ignore')
    y_tr_enc = te_train[TARGET_COL].values
    w_tr_enc = te_train['w'].values
    
    X_va_enc = te.transform(X_va)
    X_te_enc = te.transform(X_test_fe)
    
    # 对齐列
    common_cols = list(set(X_tr_enc.columns) & set(X_va_enc.columns) & set(X_te_enc.columns))
    X_tr_enc = X_tr_enc[common_cols]
    X_va_enc = X_va_enc[common_cols]
    X_te_enc = X_te_enc[common_cols]
    
    model = XGBClassifier(**best_xgb_params)
    model.fit(X_tr_enc, y_tr_enc, sample_weight=w_tr_enc, verbose=False)
    
    oof_probs[va_idx] = model.predict_proba(X_va_enc)
    test_probs += model.predict_proba(X_te_enc) / N_FOLDS
    
    fold_ba = balanced_accuracy_score(y_va, oof_probs[va_idx].argmax(axis=1))
    print(f"  Fold {fold} BA: {fold_ba:.5f}")
    
    del model
    gc.collect()

# ============================================
# 结果与提交
# ============================================
oof_ba = balanced_accuracy_score(y, oof_probs.argmax(axis=1))
print(f"\n📊 OOF Balanced Accuracy: {oof_ba:.5f}")

# 生成提交
test_preds = test_probs.argmax(axis=1)
submission = pd.DataFrame({
    'id': test_ids.values,
    TARGET_COL: [idx2label[p] for p in test_preds]
})

submission.to_csv('submission_optuna.csv', index=False)
print("\n✅ 提交文件已生成: submission_optuna.csv")
print(submission[TARGET_COL].value_counts())

Note: you may need to restart the kernel to use updated packages.
✅ Config loaded
train: (900000, 21)
test : (270000, 20)
y distribution:
Irrigation_Need
0    0.711019
1    0.265638
2    0.023343
Name: proportion, dtype: float64


IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer